# T-Social Dataset Investigation

This notebook audits the T-Social dataset for the common graph-fraud/anomaly
dataset comparison requested by Venus.

The analysis covers:

1. Actual dataset loading
2. Nodes, edges, features and class imbalance
3. Graph structure
4. Real/injected anomaly origin
5. Global heterophily
6. Local heterophily
7. Original train/validation/test protocol
8. Sources, limitations and compatibility
9. Common comparison-table row
10. Validation and export

No model-performance comparison is performed in this notebook.

For consistency with YelpChi, graph statistics use explicit edge-counting,
self-loop and reverse-edge conventions.

In [ ]:
# CELL 0 — T-SOCIAL DGL ENVIRONMENT
#
# Data analysis only: CPU build is sufficient.
# We pin Torch/DGL to matching versions.

!pip uninstall -y dgl torchdata >/dev/null 2>&1

!pip install -q --force-reinstall \
    torch==2.4.0 \
    --index-url https://download.pytorch.org/whl/cpu

!pip install -q --force-reinstall \
    "dgl==2.4.0" \
    -f https://data.dgl.ai/wheels/torch-2.4/repo.html \
    --no-deps

print("Installation finished.")
print("If Kaggle asks for a session restart, restart once.")

In [1]:
# CELL 0B — VERIFY T-SOCIAL ENVIRONMENT AFTER RESTART

import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)

try:
    import dgl
    print("DGL:", dgl.__version__)

    from dgl.data.utils import load_graphs

    print("\nPASS: DGL imported successfully.")
    print("PASS: load_graphs is available.")
    print("Environment ready for T-Social.")

except Exception as e:
    print("\nDGL IMPORT FAILED")
    print(type(e).__name__ + ":", e)

Python: 3.12.13
PyTorch: 2.4.0+cpu
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


DGL backend not selected or invalid.  Assuming PyTorch for now.


DGL: 2.4.0

PASS: DGL imported successfully.
PASS: load_graphs is available.
Environment ready for T-Social.


In [8]:
# CELL 0C — CREATE LEGACY T-SOCIAL READER ENVIRONMENT
#
# Purpose:
# Create an isolated Python 3.9 environment matching the
# original BWGNN-era dependencies:
#
#   PyTorch 1.9.0
#   DGL 0.8.1
#
# This does NOT modify the current Kaggle Python 3.12 kernel.

import os
import subprocess
import shutil


print("===== T-SOCIAL LEGACY READER SETUP =====")


# ============================================================
# 1. PATHS
# ============================================================

WORK_DIR = "/kaggle/working"

MICROMAMBA_DIR = os.path.join(
    WORK_DIR,
    "micromamba_bin"
)

MICROMAMBA = os.path.join(
    MICROMAMBA_DIR,
    "micromamba"
)

MAMBA_ROOT = os.path.join(
    WORK_DIR,
    "micromamba_root"
)

LEGACY_ENV = os.path.join(
    WORK_DIR,
    "tsocial_legacy_env"
)


os.makedirs(
    MICROMAMBA_DIR,
    exist_ok=True
)


# ============================================================
# 2. CHECK PLATFORM
# ============================================================

machine = subprocess.check_output(
    ["uname", "-m"],
    text=True
).strip()

print("Machine architecture:", machine)

if machine not in {
    "x86_64",
    "amd64"
}:
    raise RuntimeError(
        "This setup expects Kaggle's Linux x86_64 runtime.\n"
        f"Detected: {machine}"
    )


# ============================================================
# 3. DOWNLOAD MICROMAMBA IF NEEDED
# ============================================================

if not os.path.isfile(MICROMAMBA):

    print("\nDownloading micromamba...")

    command = f"""
    set -e

    cd "{MICROMAMBA_DIR}"

    curl -Ls \
      https://micro.mamba.pm/api/micromamba/linux-64/latest \
      | tar -xvj bin/micromamba

    mv bin/micromamba "{MICROMAMBA}"

    rm -rf bin

    chmod +x "{MICROMAMBA}"
    """

    subprocess.run(
        command,
        shell=True,
        check=True
    )

else:

    print(
        "\nMicromamba already exists."
    )


print(
    "Micromamba:",
    MICROMAMBA
)


# ============================================================
# 4. VERIFY MICROMAMBA
# ============================================================

result = subprocess.run(
    [
        MICROMAMBA,
        "--version"
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:

    print(result.stderr)

    raise RuntimeError(
        "Micromamba verification failed."
    )


print(
    "Micromamba version:",
    result.stdout.strip()
)


# ============================================================
# 5. CREATE PYTHON 3.9 ENVIRONMENT
# ============================================================

legacy_python = os.path.join(
    LEGACY_ENV,
    "bin",
    "python"
)


if not os.path.isfile(
    legacy_python
):

    print(
        "\nCreating isolated Python 3.9 environment..."
    )

    env = os.environ.copy()

    env[
        "MAMBA_ROOT_PREFIX"
    ] = MAMBA_ROOT

    subprocess.run(
        [
            MICROMAMBA,
            "create",
            "-y",
            "-p",
            LEGACY_ENV,
            "-c",
            "conda-forge",
            "python=3.9",
            "pip"
        ],
        env=env,
        check=True
    )

else:

    print(
        "\nLegacy Python environment already exists."
    )


legacy_pip = os.path.join(
    LEGACY_ENV,
    "bin",
    "pip"
)


# ============================================================
# 6. UPGRADE PIP TO A VERSION THAT STILL WORKS WITH OLD WHEELS
# ============================================================

print(
    "\nPreparing pip..."
)

subprocess.run(
    [
        legacy_python,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pip<25"
    ],
    check=True
)


# ============================================================
# 7. INSTALL NUMPY / SCIPY
# ============================================================

print(
    "Installing compatible NumPy / SciPy..."
)

subprocess.run(
    [
        legacy_python,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy==1.23.5",
        "scipy==1.10.1"
    ],
    check=True
)


# ============================================================
# 8. INSTALL PYTORCH 1.9.0 CPU
#
# Official PyTorch archive still provides this wheel.
# ============================================================

print(
    "Installing PyTorch 1.9.0 CPU..."
)

subprocess.run(
    [
        legacy_python,
        "-m",
        "pip",
        "install",
        "-q",
        "torch==1.9.0+cpu",
        "-f",
        "https://download.pytorch.org/whl/torch_stable.html"
    ],
    check=True
)


# ============================================================
# 9. INSTALL DGL 0.8.1 CPU
#
# DGL's official wheel repository contains:
# dgl-0.8.1-cp39-cp39-manylinux1_x86_64.whl
# ============================================================

print(
    "Installing DGL 0.8.1..."
)

subprocess.run(
    [
        legacy_python,
        "-m",
        "pip",
        "install",
        "-q",
        "dgl==0.8.1",
        "-f",
        "https://data.dgl.ai/wheels/repo.html"
    ],
    check=True
)


# ============================================================
# 10. INSTALL SMALL DEPENDENCIES USED BY OLD DGL/BWGNN
# ============================================================

print(
    "Installing supporting packages..."
)

subprocess.run(
    [
        legacy_python,
        "-m",
        "pip",
        "install",
        "-q",
        "networkx<3",
        "psutil",
        "requests",
        "tqdm"
    ],
    check=True
)


# ============================================================
# 11. VERIFY EXACT ENVIRONMENT
# ============================================================

verification_code = r'''
import sys
import numpy
import scipy
import torch
import dgl

print("Python:", sys.version.split()[0])
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("DGL:", dgl.__version__)

from dgl.data.utils import load_graphs

print("PASS: DGL import successful.")
print("PASS: load_graphs available.")
'''


verification = subprocess.run(
    [
        legacy_python,
        "-c",
        verification_code
    ],
    capture_output=True,
    text=True
)


print(
    "\n===== LEGACY ENVIRONMENT VERIFICATION ====="
)

print(
    verification.stdout
)


if verification.returncode != 0:

    print(
        "\nERROR OUTPUT:"
    )

    print(
        verification.stderr
    )

    raise RuntimeError(
        "Legacy environment verification failed."
    )


# ============================================================
# 12. FINAL PATHS FOR TASK 1
# ============================================================

print(
    "======================================"
)

print(
    "LEGACY READER ENVIRONMENT READY"
)

print(
    "======================================"
)


print(
    "\nLegacy environment:"
)

print(
    LEGACY_ENV
)


print(
    "\nLegacy Python:"
)

print(
    legacy_python
)


print(
    "\nNext step:"
)

print(
    "Run the corrected T-Social Task 1 conversion cell."
)

===== T-SOCIAL LEGACY READER SETUP =====
Machine architecture: x86_64

bin/micromamba
Micromamba: /kaggle/working/micromamba_bin/micromamba
Micromamba version: 2.9.0

Creating isolated Python 3.9 environment...
Fetch Shard Index for conda-forge/linux-64                                                      ⧖ Starting
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                        ⧖ Starting
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                           ⧖ Starting
Fetching and Parsing Packages' Shards                                                     ✔ Done (3.4 sec)

Resolving Environment                                                                           ⧖ Starting
Resolving Environment                  

warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.


Linking libzlib-1.3.2-h25fd6f3_3
Linking libgomp-16.2.0-he0feb66_4
Linking zstd-1.5.7-hb78ec9c_7
Linking _openmp_mutex-4.5-20_gnu
Linking ld_impl_linux-64-2.46.1-default_hbd61a6d_102
Linking libgcc-16.2.0-ha9f2e26_4
Linking libstdcxx-16.2.0-h934c35e_4
Linking bzip2-1.0.8-hda65f42_10
Linking libnsl-2.0.1-hb9d3cd8_1
Linking libexpat-2.8.1-hecca717_1
Linking libuuid-2.42.3-hcfc3c73_0
Linking libffi-3.7.0-h81df57d_1
Linking libxcrypt-4.4.38-h280c20c_0
Linking liblzma-5.8.3-hb03c661_1
Linking ncurses-6.6-hdb14827_1
Linking tk-8.6.13-noxft_h1df4ec4_4
Linking icu-78.3-py310h44b86e0_2
Linking readline-8.3-hd6e31c0_1
Linking libsqlite-3.53.4-h13e7031_1
Linking tzdata-2026c-h151e31d_0
Linking ca-certificates-2026.7.22-hbd8a1cb_0
Linking openssl-3.6.4-h781a0a9_0
Linking python-3.9.23-hc30ae73_0_cpython
Linking wheel-0.45.1-pyhd8ed1ab_1
Linking setuptools-80.9.0-pyhff2d567_0
Linking pip-25.2-pyh8b19718_0

Transaction finished


To activate this environment, use:

    micromamba activate /kaggle/wo

## Task 1 — Load and verify the official T-Social dataset

Venus asked us to analyse the actual dataset and calculate its characteristics
ourselves rather than relying only on statistics reported in papers.

The T-Social dataset is obtained directly from the official data release
provided by the BWGNN authors.

Official archive:

`tsocial.zip`

Official extracted graph used:

`/kaggle/working/bwgnn_official_data/dataset/tsocial_extracted/tsocial`

A separate Kaggle mirror of T-Social was also inspected. The Kaggle mirror and
official authors' file have different file sizes and different SHA256 hashes,
and the Kaggle copy could not be deserialized successfully. Therefore, the
Kaggle mirror is not used in this analysis.

The official graph loads successfully using an isolated legacy environment
compatible with the original BWGNN implementation:

- Python 3.9
- PyTorch 1.9.0
- DGL 0.8.1

The original graph is preserved unchanged.

This task:

1. loads the official T-Social graph,
2. records the actual node and stored-edge counts,
3. inspects graph schema and node/edge fields,
4. verifies the feature and label representations,
5. calculates the actual normal and anomaly node counts,
6. exports labels and stored edge endpoints into version-independent NumPy
   files,
7. preserves the original DGL edge-ID ordering,
8. verifies the exported data against the source graph.

The official graph contains 146,211,016 stored DGL edge entries. This is exactly
twice the commonly reported 73,105,508 edge count. No assumption is made at
this stage that these represent reciprocal copies of 73,105,508 undirected
edges. Reciprocity, self-loops, duplicate structure and the final comparable
undirected edge count will be checked separately during the structural and
heterophily analysis.

Values calculated from the official loaded graph are treated as authoritative.

In [11]:
# CELL 1B — EXTRACT + VERIFY OFFICIAL T-SOCIAL
#
# Uses the tsocial.zip already downloaded from the authors' Google Drive.
#
# Goals:
# 1. inspect the archive,
# 2. extract the official serialized graph,
# 3. compare it against the Kaggle mirror,
# 4. test it with the original-compatible DGL 0.8.1 environment.

import os
import zipfile
import hashlib
import subprocess


print("===== T-SOCIAL OFFICIAL ARCHIVE CHECK =====")


# ============================================================
# 1. PATHS
# ============================================================

OFFICIAL_ZIP = (
    "/kaggle/working/"
    "bwgnn_official_data/"
    "dataset/"
    "tsocial.zip"
)

KAGGLE_TSOCIAL = (
    "/kaggle/input/datasets/"
    "andrewtaj/tsocial-tfinance/"
    "tsocial"
)

LEGACY_PYTHON = (
    "/kaggle/working/"
    "tsocial_legacy_env/"
    "bin/python"
)

EXTRACT_DIR = (
    "/kaggle/working/"
    "bwgnn_official_data/"
    "dataset/"
    "tsocial_extracted"
)


for path, name in [
    (OFFICIAL_ZIP, "Official tsocial.zip"),
    (KAGGLE_TSOCIAL, "Kaggle mirror"),
    (LEGACY_PYTHON, "Legacy Python")
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )


print("\nOfficial archive:")
print(OFFICIAL_ZIP)

print(
    "Archive size:",
    f"{os.path.getsize(OFFICIAL_ZIP):,}",
    "bytes"
)


# ============================================================
# 2. INSPECT ZIP CONTENTS BEFORE EXTRACTION
# ============================================================

print("\n===== ZIP CONTENTS =====")

with zipfile.ZipFile(
    OFFICIAL_ZIP,
    "r"
) as z:

    bad_member = z.testzip()

    if bad_member is not None:
        raise RuntimeError(
            f"Corrupted ZIP member detected: {bad_member}"
        )

    names = z.namelist()

    print(
        "Number of archive entries:",
        len(names)
    )

    for name in names:
        info = z.getinfo(name)

        print(
            f"{name} | "
            f"uncompressed={info.file_size:,} bytes"
        )


print("\nPASS: Official ZIP integrity check passed.")


# ============================================================
# 3. EXTRACT OFFICIAL ARCHIVE
# ============================================================

os.makedirs(
    EXTRACT_DIR,
    exist_ok=True
)


print("\nExtracting official T-Social archive...")


with zipfile.ZipFile(
    OFFICIAL_ZIP,
    "r"
) as z:

    z.extractall(
        EXTRACT_DIR
    )


print("PASS: Extraction complete.")


# ============================================================
# 4. LOCATE EXTRACTED SERIALIZED GRAPH
# ============================================================

all_extracted_files = []

for root, dirs, files in os.walk(
    EXTRACT_DIR
):
    for filename in files:

        path = os.path.join(
            root,
            filename
        )

        all_extracted_files.append(
            path
        )


print("\n===== EXTRACTED FILES =====")

for path in all_extracted_files:

    print(
        path,
        f"({os.path.getsize(path):,} bytes)"
    )


if len(all_extracted_files) == 0:
    raise RuntimeError(
        "Archive extracted but contained no files."
    )


# ------------------------------------------------------------
# Prefer an exact basename "tsocial".
# If archive uses another internal name, use the largest file,
# because the serialized graph should be by far the largest.
# ------------------------------------------------------------

exact_candidates = [
    p for p in all_extracted_files
    if os.path.basename(p).lower() == "tsocial"
]


if len(exact_candidates) >= 1:

    OFFICIAL_TSOCIAL = max(
        exact_candidates,
        key=os.path.getsize
    )

else:

    OFFICIAL_TSOCIAL = max(
        all_extracted_files,
        key=os.path.getsize
    )

    print(
        "\nNOTE: No exact basename 'tsocial' was found."
    )

    print(
        "Using the largest extracted file as "
        "the graph candidate."
    )


print("\nOfficial extracted T-Social:")
print(OFFICIAL_TSOCIAL)

print(
    "Official extracted size:",
    f"{os.path.getsize(OFFICIAL_TSOCIAL):,}",
    "bytes"
)


# ============================================================
# 5. SHA256 FUNCTION
# ============================================================

def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


# ============================================================
# 6. COMPARE OFFICIAL FILE WITH KAGGLE MIRROR
# ============================================================

print(
    "\n===== OFFICIAL VS KAGGLE MIRROR ====="
)

official_size = os.path.getsize(
    OFFICIAL_TSOCIAL
)

kaggle_size = os.path.getsize(
    KAGGLE_TSOCIAL
)


print(
    "Official size:",
    f"{official_size:,}"
)

print(
    "Kaggle size:",
    f"{kaggle_size:,}"
)

print(
    "\nCalculating SHA256..."
)


official_hash = sha256_file(
    OFFICIAL_TSOCIAL
)

kaggle_hash = sha256_file(
    KAGGLE_TSOCIAL
)


print("\nOfficial SHA256:")
print(official_hash)

print("\nKaggle SHA256:")
print(kaggle_hash)


same_size = (
    official_size
    == kaggle_size
)

same_hash = (
    official_hash
    == kaggle_hash
)


print("\nSame size:", same_size)
print("Same SHA256:", same_hash)


if same_hash:

    print(
        "\nRESULT: Kaggle mirror is byte-for-byte "
        "identical to the official authors' file."
    )

else:

    print(
        "\nRESULT: Kaggle mirror is NOT byte-for-byte "
        "identical to the official authors' file."
    )


# ============================================================
# 7. TEST OFFICIAL FILE USING ORIGINAL-COMPATIBLE DGL
# ============================================================

TEST_SCRIPT = (
    "/kaggle/working/"
    "test_official_tsocial.py"
)


test_code = r'''
import sys
import torch
import dgl

from dgl.data.utils import load_graphs


path = sys.argv[1]


print("===== OFFICIAL T-SOCIAL LOAD TEST =====")

print(
    "Python:",
    sys.version.split()[0]
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "DGL:",
    dgl.__version__
)

print("\nLoading:")
print(path)


graphs, metadata = load_graphs(
    path
)


if len(graphs) == 0:
    raise RuntimeError(
        "No graph found."
    )


g = graphs[0]


print(
    "\nPASS: load_graphs succeeded."
)

print(
    "Graphs in file:",
    len(graphs)
)

print("\nGraph:")
print(g)

print(
    "\nNodes:",
    g.num_nodes()
)

print(
    "Stored edges:",
    g.num_edges()
)

print(
    "Node fields:",
    list(g.ndata.keys())
)

print(
    "Edge fields:",
    list(g.edata.keys())
)


if "label" in g.ndata:
    print(
        "Label shape:",
        tuple(
            g.ndata["label"].shape
        )
    )


if "feature" in g.ndata:
    print(
        "Feature shape:",
        tuple(
            g.ndata["feature"].shape
        )
    )
'''


with open(
    TEST_SCRIPT,
    "w"
) as f:

    f.write(
        test_code
    )


print(
    "\n===== TESTING OFFICIAL GRAPH ====="
)


process = subprocess.run(
    [
        LEGACY_PYTHON,
        TEST_SCRIPT,
        OFFICIAL_TSOCIAL
    ],
    capture_output=True,
    text=True
)


print(process.stdout)


if process.returncode != 0:

    print(
        "\n===== OFFICIAL GRAPH LOAD ERROR ====="
    )

    print(
        process.stderr
    )

    print(
        "\nRESULT: Official extracted file did NOT "
        "load successfully."
    )

else:

    print(
        "\nRESULT: Official extracted file loaded "
        "successfully."
    )


# ============================================================
# 8. SAVE EXACT OFFICIAL PATH FOR FINAL TASK 1
# ============================================================

PATH_RECORD = (
    "/kaggle/working/"
    "tsocial_official_path.txt"
)


with open(
    PATH_RECORD,
    "w"
) as f:

    f.write(
        OFFICIAL_TSOCIAL
    )


print("\n======================================")
print("T-SOCIAL SOURCE CHECK COMPLETE")
print("======================================")

print("\nOfficial extracted graph:")
print(OFFICIAL_TSOCIAL)

print("\nSame SHA256 as Kaggle mirror:")
print(same_hash)

print("\nOfficial load return code:")
print(process.returncode)

===== T-SOCIAL OFFICIAL ARCHIVE CHECK =====

Official archive:
/kaggle/working/bwgnn_official_data/dataset/tsocial.zip
Archive size: 744,239,223 bytes

===== ZIP CONTENTS =====
Number of archive entries: 1
tsocial | uncompressed=4,110,300,257 bytes

PASS: Official ZIP integrity check passed.

Extracting official T-Social archive...
PASS: Extraction complete.

===== EXTRACTED FILES =====
/kaggle/working/bwgnn_official_data/dataset/tsocial_extracted/tsocial (4,110,300,257 bytes)

Official extracted T-Social:
/kaggle/working/bwgnn_official_data/dataset/tsocial_extracted/tsocial
Official extracted size: 4,110,300,257 bytes

===== OFFICIAL VS KAGGLE MIRROR =====
Official size: 4,110,300,257
Kaggle size: 2,218,311,680

Calculating SHA256...

Official SHA256:
8d577114cff12f7de35eda2974f17825ef9febe20c661e95e8d21072a6dfc8d0

Kaggle SHA256:
1805a980a8cbb14e57b6dadb1916887c969707cb27bd25be1d0d5f9abf438c85

Same size: False
Same SHA256: False

RESULT: Kaggle mirror is NOT byte-for-byte identical 

In [12]:
# CELL 1 — T-SOCIAL TASK 1
# LOAD, VERIFY AND EXPORT THE OFFICIAL T-SOCIAL GRAPH

import os
import json
import subprocess

import numpy as np
import torch


print("===== T-SOCIAL TASK 1 =====")


# ============================================================
# 1. OFFICIAL SOURCE
# ============================================================

TSOCIAL_PATH = (
    "/kaggle/working/"
    "bwgnn_official_data/"
    "dataset/"
    "tsocial_extracted/"
    "tsocial"
)

LEGACY_PYTHON = (
    "/kaggle/working/"
    "tsocial_legacy_env/"
    "bin/"
    "python"
)


print("\nOfficial T-Social graph:")
print(TSOCIAL_PATH)


if not os.path.isfile(TSOCIAL_PATH):
    raise FileNotFoundError(
        "Official extracted T-Social graph was not found:\n"
        f"{TSOCIAL_PATH}"
    )


if not os.path.isfile(LEGACY_PYTHON):
    raise FileNotFoundError(
        "Legacy Python environment was not found:\n"
        f"{LEGACY_PYTHON}"
    )


source_size = os.path.getsize(
    TSOCIAL_PATH
)


print("\nPASS: Official graph exists.")

print(
    "Source file size:",
    f"{source_size:,}",
    "bytes"
)


# ============================================================
# 2. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


LABELS_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_labels.npy"
)

SRC_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_src.npy"
)

DST_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_dst.npy"
)

METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task1_metadata.json"
)


# ============================================================
# 3. LEGACY READER SCRIPT
# ============================================================

READER_SCRIPT = (
    "/kaggle/working/"
    "tsocial_task1_legacy_reader.py"
)


reader_code = r'''
import os
import sys
import json

import numpy as np
import torch
import dgl

from dgl.data.utils import load_graphs


SOURCE = (
    "/kaggle/working/"
    "bwgnn_official_data/"
    "dataset/"
    "tsocial_extracted/"
    "tsocial"
)

OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


print("===== OFFICIAL T-SOCIAL LEGACY READER =====")


# ============================================================
# A. ENVIRONMENT
# ============================================================

print("\n===== ENVIRONMENT =====")

print(
    "Python:",
    sys.version.split()[0]
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "DGL:",
    dgl.__version__
)

print(
    "NumPy:",
    np.__version__
)


# ============================================================
# B. LOAD OFFICIAL GRAPH
# ============================================================

print("\nLoading official T-Social graph...")


graphs, graph_file_metadata = load_graphs(
    SOURCE
)


if len(graphs) != 1:
    raise RuntimeError(
        f"Expected exactly one graph, found {len(graphs)}."
    )


g = graphs[0]


print(
    "\nPASS: Official graph loaded successfully."
)


# ============================================================
# C. BASIC GRAPH INFORMATION
# ============================================================

N = int(
    g.num_nodes()
)

E = int(
    g.num_edges()
)


print("\n===== RAW GRAPH =====")

print(g)

print(
    "\nNodes:",
    f"{N:,}"
)

print(
    "Raw stored DGL edge entries:",
    f"{E:,}"
)


# ============================================================
# D. GRAPH SCHEMA
# ============================================================

node_fields = list(
    g.ndata.keys()
)

edge_fields = list(
    g.edata.keys()
)

node_types = list(
    g.ntypes
)

edge_types = list(
    g.etypes
)

canonical_etypes = [
    list(x)
    for x in g.canonical_etypes
]


print("\n===== GRAPH SCHEMA =====")

print(
    "Node fields:",
    node_fields
)

print(
    "Edge fields:",
    edge_fields
)

print(
    "Node types:",
    node_types
)

print(
    "Edge types:",
    edge_types
)

print(
    "Canonical edge types:",
    canonical_etypes
)


# ============================================================
# E. REQUIRED DATA
# ============================================================

if "label" not in g.ndata:
    raise KeyError(
        "Missing required node field: label"
    )


if "feature" not in g.ndata:
    raise KeyError(
        "Missing required node field: feature"
    )


raw_labels = (
    g.ndata["label"]
)

features = (
    g.ndata["feature"]
)


print(
    "\n===== RAW LABEL / FEATURE FORMAT ====="
)

print(
    "Label shape:",
    tuple(raw_labels.shape)
)

print(
    "Label dtype:",
    raw_labels.dtype
)

print(
    "Feature shape:",
    tuple(features.shape)
)

print(
    "Feature dtype:",
    features.dtype
)


# ============================================================
# F. NORMALIZE LABELS
# ============================================================

labels = (
    raw_labels
    .long()
    .squeeze(-1)
)


if labels.ndim != 1:
    raise ValueError(
        "Expected one label per node after squeeze(-1), "
        f"found shape {tuple(labels.shape)}."
    )


if len(labels) != N:
    raise ValueError(
        "Label count does not equal node count."
    )


unique_labels, label_counts = torch.unique(
    labels,
    return_counts=True
)


if set(
    unique_labels.tolist()
) != {0, 1}:
    raise ValueError(
        "Expected binary labels {0,1}. "
        f"Found {unique_labels.tolist()}."
    )


print(
    "\n===== ACTUAL LABEL DISTRIBUTION ====="
)


label_distribution = {}


for lab, count in zip(
    unique_labels.tolist(),
    label_counts.tolist()
):

    label_distribution[
        str(int(lab))
    ] = int(count)

    print(
        f"Label {lab}: "
        f"{count:,}"
    )


# ============================================================
# G. FEATURE VERIFICATION
# ============================================================

if features.ndim != 2:
    raise ValueError(
        "Expected a 2-D feature matrix."
    )


if int(
    features.shape[0]
) != N:
    raise ValueError(
        "Feature rows do not equal node count."
    )


feature_dimension = int(
    features.shape[1]
)


print(
    "\nFeature dimension:",
    feature_dimension
)


# ============================================================
# H. SAVE LABEL ARRAY
#
# Labels contain only 0 and 1.
# int8 is lossless.
# ============================================================

labels_np = (
    labels
    .cpu()
    .numpy()
    .astype(
        np.int8,
        copy=False
    )
)


np.save(
    os.path.join(
        OUTPUT_DIR,
        "tsocial_labels.npy"
    ),
    labels_np
)


print(
    "\nPASS: Labels exported."
)


# ============================================================
# I. CREATE MEMORY-MAPPED EDGE ARRAYS
#
# Maximum node ID < 5.8 million.
# int32 therefore stores all node IDs losslessly.
# ============================================================

src_path = os.path.join(
    OUTPUT_DIR,
    "tsocial_src.npy"
)

dst_path = os.path.join(
    OUTPUT_DIR,
    "tsocial_dst.npy"
)


src_mm = np.lib.format.open_memmap(
    src_path,
    mode="w+",
    dtype=np.int32,
    shape=(E,)
)

dst_mm = np.lib.format.open_memmap(
    dst_path,
    mode="w+",
    dtype=np.int32,
    shape=(E,)
)


# ============================================================
# J. EXPORT ALL STORED EDGES
#
# Preserve exact DGL edge-ID ordering.
# ============================================================

CHUNK_SIZE = 1_000_000

num_chunks = (
    E
    + CHUNK_SIZE
    - 1
) // CHUNK_SIZE


print(
    "\n===== EDGE EXPORT ====="
)

print(
    "Stored edge entries:",
    f"{E:,}"
)

print(
    "Chunk size:",
    f"{CHUNK_SIZE:,}"
)

print(
    "Number of chunks:",
    num_chunks
)


for chunk_number, start in enumerate(
    range(
        0,
        E,
        CHUNK_SIZE
    ),
    start=1
):

    end = min(
        start + CHUNK_SIZE,
        E
    )


    eids = torch.arange(
        start,
        end,
        dtype=torch.int64
    )


    src, dst = g.find_edges(
        eids
    )


    src_mm[
        start:end
    ] = (
        src
        .cpu()
        .numpy()
        .astype(
            np.int32,
            copy=False
        )
    )


    dst_mm[
        start:end
    ] = (
        dst
        .cpu()
        .numpy()
        .astype(
            np.int32,
            copy=False
        )
    )


    if (
        chunk_number % 10 == 0
        or
        chunk_number == num_chunks
    ):

        print(
            f"Chunk "
            f"{chunk_number}/"
            f"{num_chunks} complete"
        )


src_mm.flush()
dst_mm.flush()


del src_mm
del dst_mm


print(
    "\nPASS: All stored edge endpoints exported."
)


# ============================================================
# K. METADATA
# ============================================================

metadata = {

    "dataset":
        "T-Social",

    "source":
        "Official BWGNN authors' release",

    "source_file":
        SOURCE,

    "source_file_size_bytes":
        int(
            os.path.getsize(SOURCE)
        ),

    "environment": {

        "python":
            sys.version.split()[0],

        "pytorch":
            torch.__version__,

        "dgl":
            dgl.__version__,

        "numpy":
            np.__version__
    },

    "nodes":
        N,

    "raw_stored_edge_entries":
        E,

    "feature_dimension":
        feature_dimension,

    "feature_dtype":
        str(features.dtype),

    "label_dtype":
        str(raw_labels.dtype),

    "raw_label_shape":
        list(raw_labels.shape),

    "label_distribution":
        label_distribution,

    "node_fields":
        node_fields,

    "edge_fields":
        edge_fields,

    "node_types":
        node_types,

    "edge_types":
        edge_types,

    "canonical_etypes":
        canonical_etypes,

    "export": {

        "label_dtype":
            "int8",

        "edge_endpoint_dtype":
            "int32",

        "edge_order":
            "original DGL edge-ID order",

        "chunk_size":
            CHUNK_SIZE
    }
}


with open(
    os.path.join(
        OUTPUT_DIR,
        "tsocial_task1_metadata.json"
    ),
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )


print(
    "\nPASS: Task 1 metadata saved."
)


print(
    "\n======================================"
)

print(
    "OFFICIAL GRAPH EXPORT COMPLETE"
)

print(
    "======================================"
)
'''


with open(
    READER_SCRIPT,
    "w"
) as f:

    f.write(
        reader_code
    )


print(
    "\nLegacy Task 1 reader created."
)


# ============================================================
# 4. RUN LEGACY READER
# ============================================================

print(
    "\nStarting official T-Social extraction..."
)

print(
    "The official graph contains "
    "146,211,016 stored DGL edge entries."
)

print(
    "This may take several minutes."
)


process = subprocess.Popen(
    [
        LEGACY_PYTHON,
        READER_SCRIPT
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


for line in process.stdout:

    print(
        line,
        end=""
    )


return_code = process.wait()


if return_code != 0:
    raise RuntimeError(
        "Official T-Social Task 1 reader failed "
        f"with return code {return_code}."
    )


# ============================================================
# 5. VERIFY OUTPUT FILES
# ============================================================

required_files = [
    LABELS_PATH,
    SRC_PATH,
    DST_PATH,
    METADATA_PATH
]


for path in required_files:

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Required Task 1 output missing:\n{path}"
        )


print(
    "\nPASS: All Task 1 output files exist."
)


# ============================================================
# 6. LOAD METADATA
# ============================================================

with open(
    METADATA_PATH,
    "r"
) as f:

    task1_metadata = json.load(
        f
    )


num_nodes = int(
    task1_metadata[
        "nodes"
    ]
)

raw_stored_edges = int(
    task1_metadata[
        "raw_stored_edge_entries"
    ]
)

num_features = int(
    task1_metadata[
        "feature_dimension"
    ]
)


# ============================================================
# 7. LOAD VERSION-INDEPENDENT ARRAYS
# ============================================================

labels_np = np.load(
    LABELS_PATH,
    mmap_mode="r"
)

src_edges = np.load(
    SRC_PATH,
    mmap_mode="r"
)

dst_edges = np.load(
    DST_PATH,
    mmap_mode="r"
)


# Keep labels in RAM as well because int8 uses only ~5.8 MB.

labels_np_ram = np.array(
    labels_np,
    dtype=np.int8,
    copy=True
)


labels = torch.from_numpy(
    labels_np_ram
).long()


# ============================================================
# 8. LABEL COUNTS
# ============================================================

normal_count_raw = int(
    np.count_nonzero(
        labels_np_ram == 0
    )
)

anomaly_count_raw = int(
    np.count_nonzero(
        labels_np_ram == 1
    )
)


# ============================================================
# 9. OUTPUT INTEGRITY
# ============================================================

assert (
    len(labels_np)
    == num_nodes
)

assert (
    len(src_edges)
    == raw_stored_edges
)

assert (
    len(dst_edges)
    == raw_stored_edges
)

assert (
    normal_count_raw
    + anomaly_count_raw
    == num_nodes
)


src_min = int(
    src_edges.min()
)

src_max = int(
    src_edges.max()
)

dst_min = int(
    dst_edges.min()
)

dst_max = int(
    dst_edges.max()
)


assert 0 <= src_min < num_nodes
assert 0 <= src_max < num_nodes
assert 0 <= dst_min < num_nodes
assert 0 <= dst_max < num_nodes


print(
    "\nPASS: Exported arrays passed integrity checks."
)


# ============================================================
# 10. TASK 1 FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 1 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nSource:"
)

print(
    "Official BWGNN authors' release"
)


print(
    "\nOfficial graph file:"
)

print(
    TSOCIAL_PATH
)


print(
    "\nNodes:",
    f"{num_nodes:,}"
)

print(
    "Raw stored DGL edge entries:",
    f"{raw_stored_edges:,}"
)

print(
    "Features:",
    num_features
)

print(
    "Normal nodes (label 0):",
    f"{normal_count_raw:,}"
)

print(
    "Anomaly nodes (label 1):",
    f"{anomaly_count_raw:,}"
)


print(
    "\nNode fields:",
    task1_metadata[
        "node_fields"
    ]
)

print(
    "Edge fields:",
    task1_metadata[
        "edge_fields"
    ]
)


print(
    "\nFeature dtype:",
    task1_metadata[
        "feature_dtype"
    ]
)

print(
    "Original label dtype:",
    task1_metadata[
        "label_dtype"
    ]
)


print(
    "\nIMPORTANT EDGE NOTE:"
)

print(
    "146,211,016 is the number of stored DGL "
    "edge entries in the official graph."
)

print(
    "The commonly reported 73,105,508 count "
    "is exactly half of this value."
)

print(
    "We will NOT assume why until Task 5 verifies "
    "reciprocity, self-loops and duplicate structure."
)


print(
    "\nTask 1 output directory:"
)

print(
    OUTPUT_DIR
)


print(
    "\nDo not continue to Task 2 until "
    "this output has been checked."
)

===== T-SOCIAL TASK 1 =====

Official T-Social graph:
/kaggle/working/bwgnn_official_data/dataset/tsocial_extracted/tsocial

PASS: Official graph exists.
Source file size: 4,110,300,257 bytes

Legacy Task 1 reader created.

Starting official T-Social extraction...
The official graph contains 146,211,016 stored DGL edge entries.
This may take several minutes.
===== OFFICIAL T-SOCIAL LEGACY READER =====

===== ENVIRONMENT =====
Python: 3.9.23
PyTorch: 1.9.0+cpu
DGL: 0.8.1
NumPy: 1.23.5

Loading official T-Social graph...

PASS: Official graph loaded successfully.

===== RAW GRAPH =====
Graph(num_nodes=5781065, num_edges=146211016,
      ndata_schemes={'feature': Scheme(shape=(10,), dtype=torch.int64), 'label': Scheme(shape=(), dtype=torch.int64), '_ID': Scheme(shape=(), dtype=torch.int64)}
      edata_schemes={'_ID': Scheme(shape=(), dtype=torch.int64)})

Nodes: 5,781,065
Raw stored DGL edge entries: 146,211,016

===== GRAPH SCHEMA =====
Node fields: ['feature', 'label', '_ID']
Edge fiel

## Task 2 — Calculate numerical dataset characteristics

This step calculates the main numerical characteristics of T-Social directly
from the labels extracted from the official graph.

For consistency with the other datasets in the benchmark, we record:

- number of nodes,
- raw stored DGL edge entries,
- number of node features,
- number of normal nodes,
- number of anomaly nodes,
- anomaly percentage,
- normal-to-anomaly imbalance ratio.

Label `0` is treated as the normal class and label `1` as the anomaly class.

The edge value reported in this task is the raw number of stored DGL edge
entries. The final comparable undirected edge count will not be determined
until the graph structure is checked for reciprocity, self-loops and duplicate
edges in the later structural analysis.

In [13]:
# CELL 2 — T-SOCIAL TASK 2:
# NUMERICAL DATASET CHARACTERISTICS

import numpy as np


print("===== T-SOCIAL TASK 2 =====")


# ============================================================
# 1. BASIC VALUES FROM TASK 1
# ============================================================

num_nodes = int(
    len(labels_np_ram)
)

num_features = int(
    task1_metadata[
        "feature_dimension"
    ]
)

raw_stored_edges = int(
    len(src_edges)
)


# ============================================================
# 2. CLASS COUNTS
# ============================================================

normal_count = int(
    np.count_nonzero(
        labels_np_ram == 0
    )
)

anomaly_count = int(
    np.count_nonzero(
        labels_np_ram == 1
    )
)


# ============================================================
# 3. VERIFY BINARY LABELS
# ============================================================

unique_labels = np.unique(
    labels_np_ram
)


assert set(
    unique_labels.tolist()
) == {0, 1}, (
    f"Unexpected labels found: "
    f"{unique_labels.tolist()}"
)


assert (
    normal_count
    + anomaly_count
    == num_nodes
), (
    "Normal + anomaly counts do not "
    "equal total node count."
)


print(
    "\nPASS: Binary labels verified."
)

print(
    "PASS: Class counts sum to total nodes."
)


# ============================================================
# 4. ANOMALY PERCENTAGE
# ============================================================

anomaly_percentage = (
    anomaly_count
    / num_nodes
    * 100.0
)


# ============================================================
# 5. IMBALANCE RATIO
#
# Convention:
# Normal : Anomaly
# ============================================================

if anomaly_count == 0:

    raise ZeroDivisionError(
        "Anomaly count is zero; imbalance ratio "
        "cannot be calculated."
    )


imbalance_ratio = (
    normal_count
    / anomaly_count
)


# ============================================================
# 6. TASK 2 RESULTS
# ============================================================

print(
    "\n===== NUMERICAL CHARACTERISTICS ====="
)


print(
    "Nodes:",
    f"{num_nodes:,}"
)

print(
    "Raw stored DGL edge entries:",
    f"{raw_stored_edges:,}"
)

print(
    "Node features:",
    num_features
)

print(
    "Normal nodes (label 0):",
    f"{normal_count:,}"
)

print(
    "Anomaly nodes (label 1):",
    f"{anomaly_count:,}"
)

print(
    "Anomaly percentage:",
    f"{anomaly_percentage:.6f}%"
)

print(
    "Normal : Anomaly imbalance ratio:",
    f"{imbalance_ratio:.6f}:1"
)


# ============================================================
# 7. SOURCE CONSISTENCY CHECK
# ============================================================

expected_nodes = 5_781_065

expected_normal = 5_606_785

expected_anomaly = 174_280


assert (
    num_nodes
    == expected_nodes
)

assert (
    normal_count
    == expected_normal
)

assert (
    anomaly_count
    == expected_anomaly
)


print(
    "\nPASS: Calculated class counts match "
    "the actual Task 1 graph output."
)


# ============================================================
# 8. STORE TASK 2 RESULTS
# ============================================================

task2_results = {

    "dataset":
        "T-Social",

    "nodes":
        num_nodes,

    "raw_stored_edge_entries":
        raw_stored_edges,

    "features":
        num_features,

    "normal_nodes":
        normal_count,

    "anomaly_nodes":
        anomaly_count,

    "anomaly_percentage":
        float(
            anomaly_percentage
        ),

    "normal_to_anomaly_ratio":
        float(
            imbalance_ratio
        )
}


# ============================================================
# 9. FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 2 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nValues to record:"
)

print(
    f"Nodes = {num_nodes:,}"
)

print(
    f"Raw stored edges = "
    f"{raw_stored_edges:,}"
)

print(
    f"Features = {num_features}"
)

print(
    f"Normal nodes = {normal_count:,}"
)

print(
    f"Anomaly nodes = {anomaly_count:,}"
)

print(
    f"Anomaly % = "
    f"{anomaly_percentage:.6f}%"
)

print(
    f"Imbalance = "
    f"{imbalance_ratio:.6f}:1 "
    f"(Normal:Anomaly)"
)

print(
    "\nEDGE NOTE:"
)

print(
    "Raw stored edge entries are still "
    "146,211,016."
)

print(
    "Do not replace this with 73,105,508 "
    "until structural verification is complete."
)

===== T-SOCIAL TASK 2 =====

PASS: Binary labels verified.
PASS: Class counts sum to total nodes.

===== NUMERICAL CHARACTERISTICS =====
Nodes: 5,781,065
Raw stored DGL edge entries: 146,211,016
Node features: 10
Normal nodes (label 0): 5,606,785
Anomaly nodes (label 1): 174,280
Anomaly percentage: 3.014669%
Normal : Anomaly imbalance ratio: 32.171133:1

PASS: Calculated class counts match the actual Task 1 graph output.

T-SOCIAL TASK 2 COMPLETE

Values to record:
Nodes = 5,781,065
Raw stored edges = 146,211,016
Features = 10
Normal nodes = 5,606,785
Anomaly nodes = 174,280
Anomaly % = 3.014669%
Imbalance = 32.171133:1 (Normal:Anomaly)

EDGE NOTE:
Raw stored edge entries are still 146,211,016.
Do not replace this with 73,105,508 until structural verification is complete.


## Task 3 — Identify graph type, node type, relation type and temporal structure

This step identifies the structural characteristics of T-Social.

The structural schema was verified directly from the official T-Social graph
during Task 1:

- DGL node types: `['_U']`
- DGL edge types: `['_V']`
- Canonical relation: `['_U', '_V', '_U']`

The internal names `_U` and `_V` are DGL identifiers and are not the semantic
meaning of the nodes or edges.

According to the T-Social dataset description:

- **Node type:** anonymized social-network account
- **Relation type:** social friendship
- **Edge meaning:** friendship maintained for more than three months

Because the released benchmark is represented as one graph rather than a
sequence of timestamped graph snapshots, it is treated as a **static graph**.

The three-month friendship rule is an edge-selection condition and does not
make the released benchmark a dynamic graph.

In [4]:
# CELL 3 — T-SOCIAL TASK 3
# GRAPH TYPE, NODE TYPE, RELATION TYPE AND TEMPORAL STRUCTURE
#
# NOTE:
# Kaggle cleared /kaggle/working after the session changed.
# Therefore this cell uses the graph schema that was already
# verified directly from the official T-Social graph in Task 1.
#
# It does NOT require the 146M-edge graph to be loaded again.

import os
import json


print("===== T-SOCIAL TASK 3 =====")


# ============================================================
# 1. VERIFIED TASK 1 GRAPH SCHEMA
# ============================================================

# These values were obtained directly from the official
# T-Social graph during the successful Task 1 run.

node_types_dgl = [
    "_U"
]

edge_types_dgl = [
    "_V"
]

canonical_etypes_dgl = [
    [
        "_U",
        "_V",
        "_U"
    ]
]


print(
    "\n===== VERIFIED TASK 1 DGL SCHEMA ====="
)

print(
    "DGL node types:",
    node_types_dgl
)

print(
    "DGL edge types:",
    edge_types_dgl
)

print(
    "Canonical edge types:",
    canonical_etypes_dgl
)


# ============================================================
# 2. VERIFIED TASK 1 SCALE
# ============================================================

verified_nodes = (
    5_781_065
)

verified_raw_edges = (
    146_211_016
)

verified_features = (
    10
)


print(
    "\n===== VERIFIED TASK 1 SCALE ====="
)

print(
    "Nodes:",
    f"{verified_nodes:,}"
)

print(
    "Raw stored DGL edge entries:",
    f"{verified_raw_edges:,}"
)

print(
    "Features:",
    verified_features
)


# ============================================================
# 3. COUNT STRUCTURAL TYPES
# ============================================================

num_node_types = len(
    node_types_dgl
)

num_edge_types = len(
    edge_types_dgl
)

num_canonical_relations = len(
    canonical_etypes_dgl
)


print(
    "\nNumber of node types:",
    num_node_types
)

print(
    "Number of edge types:",
    num_edge_types
)

print(
    "Number of canonical relations:",
    num_canonical_relations
)


# ============================================================
# 4. VERIFY STRUCTURAL CLASSIFICATION
# ============================================================

is_homogeneous = (
    num_node_types == 1
    and
    num_canonical_relations == 1
)

is_single_relation = (
    num_edge_types == 1
    and
    num_canonical_relations == 1
)


assert is_homogeneous, (
    "Expected T-Social to contain one node type "
    "and one canonical relation."
)


assert is_single_relation, (
    "Expected T-Social to contain one relation type."
)


print(
    "\nPASS: One node type."
)

print(
    "PASS: One relation type."
)

print(
    "PASS: Homogeneous graph representation."
)

print(
    "PASS: Single-relational graph representation."
)


# ============================================================
# 5. INTERNAL DGL IDENTIFIERS
# ============================================================

internal_node_type = (
    node_types_dgl[0]
)

internal_edge_type = (
    edge_types_dgl[0]
)


print(
    "\n===== INTERNAL DGL IDENTIFIERS ====="
)

print(
    "Internal node type:",
    internal_node_type
)

print(
    "Internal edge type:",
    internal_edge_type
)


# ============================================================
# 6. SEMANTIC CLASSIFICATION
# ============================================================

graph_type = (
    "Homogeneous"
)

relation_structure = (
    "Single-relational"
)

semantic_node_type = (
    "Anonymized social-network account"
)

semantic_relation_type = (
    "Social friendship"
)

edge_definition = (
    "Friendship relationship maintained "
    "for more than three months"
)


# ============================================================
# 7. STATIC / DYNAMIC CLASSIFICATION
# ============================================================

temporal_structure = (
    "Static"
)

temporal_reason = (
    "The released benchmark is represented as one graph "
    "rather than a sequence of timestamped graph snapshots. "
    "The more-than-three-month friendship condition is an "
    "edge-selection criterion."
)


# ============================================================
# 8. DISPLAY FINAL CLASSIFICATION
# ============================================================

print(
    "\n===== STRUCTURAL CLASSIFICATION ====="
)

print(
    "Graph type:",
    graph_type
)

print(
    "Relation structure:",
    relation_structure
)

print(
    "Semantic node type:",
    semantic_node_type
)

print(
    "Semantic relation type:",
    semantic_relation_type
)

print(
    "Edge meaning:",
    edge_definition
)

print(
    "Temporal type:",
    temporal_structure
)


# ============================================================
# 9. IMPORTANT DGL INTERPRETATION
# ============================================================

print(
    "\n===== IMPORTANT INTERPRETATION ====="
)

print(
    "_U and _V are internal DGL identifiers."
)

print(
    "They are NOT the semantic node/relation names."
)

print(
    "Report node type as:"
)

print(
    semantic_node_type
)

print(
    "Report relation as:"
)

print(
    semantic_relation_type
)


# ============================================================
# 10. STORE TASK 3 RESULTS
# ============================================================

task3_results = {

    "dataset":
        "T-Social",

    "graph_type":
        graph_type,

    "relation_structure":
        relation_structure,

    "number_of_node_types":
        num_node_types,

    "number_of_relation_types":
        num_canonical_relations,

    "internal_dgl_node_type":
        internal_node_type,

    "internal_dgl_edge_type":
        internal_edge_type,

    "canonical_relation":
        canonical_etypes_dgl[0],

    "semantic_node_type":
        semantic_node_type,

    "semantic_relation_type":
        semantic_relation_type,

    "edge_definition":
        edge_definition,

    "temporal_structure":
        temporal_structure,

    "temporal_reason":
        temporal_reason,

    "task1_verified_nodes":
        verified_nodes,

    "task1_verified_raw_edge_entries":
        verified_raw_edges,

    "task1_verified_features":
        verified_features
}


# ============================================================
# 11. SAVE TASK 3 OUTPUT
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


TASK3_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task3_structure.json"
)


with open(
    TASK3_PATH,
    "w"
) as f:

    json.dump(
        task3_results,
        f,
        indent=4
    )


print(
    "\nPASS: Task 3 results saved:"
)

print(
    TASK3_PATH
)


# ============================================================
# 12. FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 3 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nValues to record:"
)

print(
    "Graph type = Homogeneous"
)

print(
    "Relation structure = Single-relational"
)

print(
    "Node type = Anonymized social-network account"
)

print(
    "Relation = Social friendship"
)

print(
    "Edge meaning = Friendship maintained "
    "for more than three months"
)

print(
    "Temporal type = Static"
)

===== T-SOCIAL TASK 3 =====

===== VERIFIED TASK 1 DGL SCHEMA =====
DGL node types: ['_U']
DGL edge types: ['_V']
Canonical edge types: [['_U', '_V', '_U']]

===== VERIFIED TASK 1 SCALE =====
Nodes: 5,781,065
Raw stored DGL edge entries: 146,211,016
Features: 10

Number of node types: 1
Number of edge types: 1
Number of canonical relations: 1

PASS: One node type.
PASS: One relation type.
PASS: Homogeneous graph representation.
PASS: Single-relational graph representation.

===== INTERNAL DGL IDENTIFIERS =====
Internal node type: _U
Internal edge type: _V

===== STRUCTURAL CLASSIFICATION =====
Graph type: Homogeneous
Relation structure: Single-relational
Semantic node type: Anonymized social-network account
Semantic relation type: Social friendship
Edge meaning: Friendship relationship maintained for more than three months
Temporal type: Static

===== IMPORTANT INTERPRETATION =====
_U and _V are internal DGL identifiers.
They are NOT the semantic node/relation names.
Report node type a

## Task 4 — Identify anomaly origin and label source

This step determines whether the anomalous nodes in T-Social are real-world
observations or synthetically injected anomalies, and how the anomaly labels
were obtained.

Tang et al. (2022) describe T-Social as one of two newly released
large-scale **real-world** graph anomaly detection datasets.

T-Social uses the same node annotations and node features as T-Finance.
The authors state that human experts annotate accounts as anomalous when they
belong to categories such as:

- fraud,
- money laundering,
- online gambling.

Therefore, the anomaly labels in the released T-Social benchmark are treated
as **real-world, human-expert annotations**, not synthetic or artificially
injected anomaly labels.

For the released graph:

- Label `0` = normal account
- Label `1` = anomalous account

From the actual official graph analysed in Tasks 1–2:

- Normal accounts: 5,606,785
- Anomalous accounts: 174,280
- Anomaly rate: 3.014669%

This classification applies to the original released T-Social dataset.
Synthetic anomaly perturbations used elsewhere in experimental analyses should
not be confused with the original T-Social node labels.

In [5]:
# CELL 4 — T-SOCIAL TASK 4
# ANOMALY ORIGIN AND LABEL SOURCE
#
# This task records the provenance and interpretation
# of the anomaly labels in the released T-Social dataset.
#
# No large graph loading is required.

import os
import json


print("===== T-SOCIAL TASK 4 =====")


# ============================================================
# 1. VERIFIED NUMERICAL RESULTS FROM TASKS 1–2
# ============================================================

num_nodes = 5_781_065

normal_nodes = 5_606_785

anomaly_nodes = 174_280

anomaly_percentage = (
    anomaly_nodes
    / num_nodes
    * 100.0
)


# ============================================================
# 2. VERIFY COUNTS
# ============================================================

assert (
    normal_nodes
    + anomaly_nodes
    == num_nodes
), (
    "Normal and anomaly counts do not "
    "sum to total nodes."
)


assert abs(
    anomaly_percentage
    - 3.014669
) < 0.000001, (
    "Anomaly percentage differs from "
    "the verified Task 2 result."
)


print(
    "\nPASS: Verified Task 2 counts loaded."
)

print(
    "Total nodes:",
    f"{num_nodes:,}"
)

print(
    "Normal nodes:",
    f"{normal_nodes:,}"
)

print(
    "Anomaly nodes:",
    f"{anomaly_nodes:,}"
)

print(
    "Anomaly percentage:",
    f"{anomaly_percentage:.6f}%"
)


# ============================================================
# 3. ANOMALY ORIGIN
# ============================================================

# Tang et al. describe T-Finance and T-Social as
# large-scale real-world graph anomaly detection datasets.
#
# Therefore the original released T-Social labels are
# NOT synthetic/injected anomaly labels.

dataset_origin = (
    "Real-world"
)

anomaly_origin = (
    "Real-world / non-injected"
)

synthetic_anomalies_in_original_dataset = (
    False
)


# ============================================================
# 4. LABEL SOURCE
# ============================================================

# The paper states:
#
# - T-Social has the same node annotations as T-Finance.
# - Human experts annotate anomalous accounts.
# - Example anomaly categories include fraud,
#   money laundering and online gambling.

label_source = (
    "Human-expert annotation"
)

annotation_scope = (
    "Account-level anomaly annotation"
)


anomaly_categories = [
    "Fraud",
    "Money laundering",
    "Online gambling"
]


# ============================================================
# 5. LABEL INTERPRETATION
# ============================================================

label_0_meaning = (
    "Normal account"
)

label_1_meaning = (
    "Anomalous account"
)


# ============================================================
# 6. IMPORTANT LIMITATION / INTERPRETATION
# ============================================================

# The positive class should generally be called
# "anomaly" rather than treating every positive node
# as one identical fraud category.
#
# The paper gives multiple examples of anomalous account
# categories.

positive_class_reporting = (
    "Anomalous account"
)

important_note = (
    "The positive class includes anomalous accounts from "
    "categories such as fraud, money laundering and online "
    "gambling. Therefore, 'anomaly' is the most accurate "
    "general label for the positive class."
)


# ============================================================
# 7. DISTINGUISH ORIGINAL LABELS FROM SYNTHETIC EXPERIMENTS
# ============================================================

synthetic_experiment_note = (
    "Synthetic anomaly perturbations discussed elsewhere "
    "in the BWGNN paper are experimental manipulations and "
    "should not be confused with the original T-Social "
    "human-expert anomaly labels."
)


# ============================================================
# 8. DISPLAY TASK 4 CLASSIFICATION
# ============================================================

print(
    "\n===== ANOMALY ORIGIN ====="
)

print(
    "Dataset origin:",
    dataset_origin
)

print(
    "Anomaly origin:",
    anomaly_origin
)

print(
    "Synthetic anomalies in original dataset:",
    synthetic_anomalies_in_original_dataset
)


print(
    "\n===== LABEL SOURCE ====="
)

print(
    "Label source:",
    label_source
)

print(
    "Annotation level:",
    annotation_scope
)

print(
    "Label 0:",
    label_0_meaning
)

print(
    "Label 1:",
    label_1_meaning
)


print(
    "\nExample anomalous-account categories:"
)

for category in anomaly_categories:

    print(
        " -",
        category
    )


# ============================================================
# 9. STORE TASK 4 RESULTS
# ============================================================

task4_results = {

    "dataset":
        "T-Social",

    "dataset_origin":
        dataset_origin,

    "anomaly_origin":
        anomaly_origin,

    "synthetic_anomalies_in_original_dataset":
        synthetic_anomalies_in_original_dataset,

    "label_source":
        label_source,

    "annotation_scope":
        annotation_scope,

    "label_0_meaning":
        label_0_meaning,

    "label_1_meaning":
        label_1_meaning,

    "positive_class_reporting":
        positive_class_reporting,

    "example_anomaly_categories":
        anomaly_categories,

    "total_nodes":
        num_nodes,

    "normal_nodes":
        normal_nodes,

    "anomaly_nodes":
        anomaly_nodes,

    "anomaly_percentage":
        float(anomaly_percentage),

    "important_note":
        important_note,

    "synthetic_experiment_note":
        synthetic_experiment_note
}


# ============================================================
# 10. SAVE RESULTS
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


TASK4_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task4_anomaly_origin.json"
)


with open(
    TASK4_PATH,
    "w"
) as f:

    json.dump(
        task4_results,
        f,
        indent=4
    )


print(
    "\nPASS: Task 4 results saved:"
)

print(
    TASK4_PATH
)


# ============================================================
# 11. FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 4 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nValues to record:"
)

print(
    "Dataset origin = Real-world"
)

print(
    "Anomaly origin = Real-world / non-injected"
)

print(
    "Label source = Human-expert annotation"
)

print(
    "Positive label = Anomalous account"
)

print(
    "Examples = Fraud, money laundering, "
    "online gambling"
)

print(
    "Original synthetic/injected anomalies = No"
)


print(
    "\nIMPORTANT:"
)

print(
    "Use 'anomaly nodes' as the general term "
    "rather than calling every positive node fraud."
)

===== T-SOCIAL TASK 4 =====

PASS: Verified Task 2 counts loaded.
Total nodes: 5,781,065
Normal nodes: 5,606,785
Anomaly nodes: 174,280
Anomaly percentage: 3.014669%

===== ANOMALY ORIGIN =====
Dataset origin: Real-world
Anomaly origin: Real-world / non-injected
Synthetic anomalies in original dataset: False

===== LABEL SOURCE =====
Label source: Human-expert annotation
Annotation level: Account-level anomaly annotation
Label 0: Normal account
Label 1: Anomalous account

Example anomalous-account categories:
 - Fraud
 - Money laundering
 - Online gambling

PASS: Task 4 results saved:
/kaggle/working/tsocial_analysis/tsocial_task4_anomaly_origin.json

T-SOCIAL TASK 4 COMPLETE

Values to record:
Dataset origin = Real-world
Anomaly origin = Real-world / non-injected
Label source = Human-expert annotation
Positive label = Anomalous account
Examples = Fraud, money laundering, online gambling
Original synthetic/injected anomalies = No

IMPORTANT:
Use 'anomaly nodes' as the general term rath

## Task 5 — Verify graph structure and calculate global heterophily

This task verifies the edge structure of the official T-Social graph before
calculating global heterophily.

The official DGL graph contains 146,211,016 stored directed edge entries,
while benchmark tables commonly report 73,105,508 T-Social edges.

We do not assume that the stored count should simply be divided by two.

Instead, the actual graph is checked for:

- self-loops,
- parallel directed edges,
- reciprocal edges,
- upper- and lower-direction edge counts,
- the final one-per-undirected-pair edge count.

For consistency with the YelpChi analysis, global heterophily is calculated on
the simple undirected graph:

H_global =
(number of edges connecting nodes with different labels)
/
(number of eligible labelled undirected edges)

All T-Social nodes have binary labels, so every non-self structural edge is
eligible.

If the graph is confirmed to be simple and fully reciprocal, one orientation
(`source < destination`) is used to represent each undirected friendship
exactly once.

This task also accumulates each node's undirected degree and number of
different-label neighbours. These arrays are saved for Task 6 so local
heterophily can be calculated without processing the 146 million stored edges
again.

In [7]:
# CELL 5 — T-SOCIAL TASK 5
# STRUCTURAL VERIFICATION + GLOBAL HETEROPHILY
#
# SELF-RECOVERING VERSION
#
# This cell:
# 1. restores the verified official T-Social file if needed,
# 2. verifies source size + SHA256,
# 3. restores the legacy DGL 0.8.1 environment if needed,
# 4. loads the official graph,
# 5. checks self-loops,
# 6. checks directed parallel edges,
# 7. checks reciprocity,
# 8. establishes the true undirected edge count,
# 9. calculates global heterophily,
# 10. prepares degree + label arrays for Task 6.

import os
import sys
import json
import hashlib
import zipfile
import subprocess
import shutil


print("===== T-SOCIAL TASK 5 =====")


# ============================================================
# 1. VERIFIED OFFICIAL SOURCE INFORMATION
# ============================================================

WORK_DIR = "/kaggle/working"

SOURCE_DIR = os.path.join(
    WORK_DIR,
    "tsocial_official_source"
)

os.makedirs(
    SOURCE_DIR,
    exist_ok=True
)


OFFICIAL_ZIP = os.path.join(
    SOURCE_DIR,
    "tsocial.zip"
)

OFFICIAL_TSOCIAL = os.path.join(
    SOURCE_DIR,
    "tsocial"
)


# Official BWGNN Google Drive file ID
OFFICIAL_FILE_ID = (
    "10nY_IwxT32KdfpcTqoy8wgckXK2xayXd"
)

OFFICIAL_DOWNLOAD_URL = (
    "https://drive.google.com/uc?id="
    + OFFICIAL_FILE_ID
)


# Values already verified from the authors' official release

EXPECTED_ZIP_SIZE = (
    744_239_223
)

EXPECTED_GRAPH_SIZE = (
    4_110_300_257
)

EXPECTED_GRAPH_SHA256 = (
    "8d577114cff12f7de35eda2974f17825"
    "ef9febe20c661e95e8d21072a6dfc8d0"
)

EXPECTED_NODES = (
    5_781_065
)

EXPECTED_STORED_EDGES = (
    146_211_016
)

EXPECTED_FEATURES = (
    10
)


# ============================================================
# 2. SHA256 HELPER
# ============================================================

def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


# ============================================================
# 3. RESTORE OFFICIAL ARCHIVE IF NEEDED
# ============================================================

if not os.path.isfile(
    OFFICIAL_TSOCIAL
):

    print(
        "\nOfficial extracted graph is not "
        "currently available."
    )

    print(
        "Recovering the authors' official "
        "T-Social release..."
    )


    # --------------------------------------------------------
    # Download ZIP only if it is not already present
    # --------------------------------------------------------

    if not os.path.isfile(
        OFFICIAL_ZIP
    ):

        print(
            "\nInstalling/updating gdown..."
        )

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                "gdown"
            ],
            check=True
        )


        print(
            "\nDownloading official tsocial.zip..."
        )

        print(
            "Google Drive source:"
        )

        print(
            OFFICIAL_DOWNLOAD_URL
        )


        # IMPORTANT:
        # Current gdown versions take URL/ID directly.
        # Do NOT use the removed --id option.

        download_result = subprocess.run(
            [
                sys.executable,
                "-m",
                "gdown",
                OFFICIAL_DOWNLOAD_URL,
                "-O",
                OFFICIAL_ZIP
            ]
        )


        if download_result.returncode != 0:

            raise RuntimeError(
                "Official T-Social download failed."
            )


    # --------------------------------------------------------
    # Verify ZIP exists
    # --------------------------------------------------------

    if not os.path.isfile(
        OFFICIAL_ZIP
    ):

        raise FileNotFoundError(
            "Official tsocial.zip was not created."
        )


    zip_size = os.path.getsize(
        OFFICIAL_ZIP
    )


    print(
        "\nOfficial ZIP size:",
        f"{zip_size:,}",
        "bytes"
    )


    if zip_size != EXPECTED_ZIP_SIZE:

        raise RuntimeError(
            "Downloaded ZIP size differs from "
            "the previously verified official archive.\n"
            f"Expected: {EXPECTED_ZIP_SIZE:,}\n"
            f"Found:    {zip_size:,}"
        )


    # --------------------------------------------------------
    # Test ZIP integrity
    # --------------------------------------------------------

    print(
        "\nChecking ZIP integrity..."
    )


    with zipfile.ZipFile(
        OFFICIAL_ZIP,
        "r"
    ) as z:

        bad_member = z.testzip()

        if bad_member is not None:

            raise RuntimeError(
                f"Corrupted ZIP member: {bad_member}"
            )


        members = z.namelist()


        print(
            "Archive contents:",
            members
        )


        if "tsocial" not in members:

            raise RuntimeError(
                "Official archive does not contain "
                "the expected file named 'tsocial'."
            )


        print(
            "\nExtracting official T-Social graph..."
        )


        z.extract(
            "tsocial",
            SOURCE_DIR
        )


    print(
        "PASS: Official archive extracted."
    )


# ============================================================
# 4. VERIFY OFFICIAL GRAPH EXISTS
# ============================================================

if not os.path.isfile(
    OFFICIAL_TSOCIAL
):

    raise FileNotFoundError(
        "Official extracted T-Social graph "
        "could not be found."
    )


print(
    "\nOfficial graph:"
)

print(
    OFFICIAL_TSOCIAL
)


graph_size = os.path.getsize(
    OFFICIAL_TSOCIAL
)


print(
    "Graph size:",
    f"{graph_size:,}",
    "bytes"
)


if graph_size != EXPECTED_GRAPH_SIZE:

    raise RuntimeError(
        "Official graph size mismatch.\n"
        f"Expected: {EXPECTED_GRAPH_SIZE:,}\n"
        f"Found:    {graph_size:,}"
    )


# ============================================================
# 5. VERIFY SHA256
# ============================================================

print(
    "\nCalculating official graph SHA256..."
)


graph_sha256 = sha256_file(
    OFFICIAL_TSOCIAL
)


print(
    "SHA256:"
)

print(
    graph_sha256
)


if graph_sha256 != EXPECTED_GRAPH_SHA256:

    raise RuntimeError(
        "Official graph SHA256 mismatch.\n"
        "Do not analyse this file."
    )


print(
    "\nPASS: Official source identity verified."
)


# ============================================================
# 6. LEGACY ENVIRONMENT PATH
# ============================================================

LEGACY_ENV = os.path.join(
    WORK_DIR,
    "tsocial_legacy_env"
)

LEGACY_PYTHON = os.path.join(
    LEGACY_ENV,
    "bin",
    "python"
)


# ============================================================
# 7. RECREATE LEGACY ENVIRONMENT IF SESSION CLEARED IT
# ============================================================

if not os.path.isfile(
    LEGACY_PYTHON
):

    print(
        "\nLegacy environment is missing."
    )

    print(
        "Recreating Python 3.9 / "
        "PyTorch 1.9 / DGL 0.8.1..."
    )


    MICROMAMBA_DIR = os.path.join(
        WORK_DIR,
        "micromamba_bin"
    )

    MICROMAMBA = os.path.join(
        MICROMAMBA_DIR,
        "micromamba"
    )

    MAMBA_ROOT = os.path.join(
        WORK_DIR,
        "micromamba_root"
    )


    os.makedirs(
        MICROMAMBA_DIR,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Install micromamba if needed
    # --------------------------------------------------------

    if not os.path.isfile(
        MICROMAMBA
    ):

        print(
            "\nDownloading micromamba..."
        )


        command = f'''
        set -e

        cd "{MICROMAMBA_DIR}"

        curl -Ls \
        https://micro.mamba.pm/api/micromamba/linux-64/latest \
        | tar -xvj bin/micromamba

        mv bin/micromamba "{MICROMAMBA}"

        rm -rf bin

        chmod +x "{MICROMAMBA}"
        '''


        subprocess.run(
            command,
            shell=True,
            check=True
        )


    env = os.environ.copy()

    env[
        "MAMBA_ROOT_PREFIX"
    ] = MAMBA_ROOT


    # --------------------------------------------------------
    # Create Python 3.9 environment
    # --------------------------------------------------------

    print(
        "\nCreating Python 3.9 environment..."
    )


    subprocess.run(
        [
            MICROMAMBA,
            "create",
            "-y",
            "-p",
            LEGACY_ENV,
            "-c",
            "conda-forge",
            "python=3.9",
            "pip"
        ],
        env=env,
        check=True
    )


    # --------------------------------------------------------
    # Install compatible NumPy / SciPy
    # --------------------------------------------------------

    print(
        "Installing numerical dependencies..."
    )


    subprocess.run(
        [
            LEGACY_PYTHON,
            "-m",
            "pip",
            "install",
            "-q",
            "pip<25",
            "numpy==1.23.5",
            "scipy==1.10.1"
        ],
        check=True
    )


    # --------------------------------------------------------
    # PyTorch 1.9 CPU
    # --------------------------------------------------------

    print(
        "Installing PyTorch 1.9.0..."
    )


    subprocess.run(
        [
            LEGACY_PYTHON,
            "-m",
            "pip",
            "install",
            "-q",
            "torch==1.9.0+cpu",
            "-f",
            "https://download.pytorch.org/whl/"
            "torch_stable.html"
        ],
        check=True
    )


    # --------------------------------------------------------
    # DGL 0.8.1 CPU
    # --------------------------------------------------------

    print(
        "Installing DGL 0.8.1..."
    )


    subprocess.run(
        [
            LEGACY_PYTHON,
            "-m",
            "pip",
            "install",
            "-q",
            "dgl==0.8.1",
            "-f",
            "https://data.dgl.ai/wheels/repo.html"
        ],
        check=True
    )


# ============================================================
# 8. VERIFY LEGACY ENVIRONMENT
# ============================================================

verification_code = r'''
import sys
import numpy
import torch
import dgl

print("Python:", sys.version.split()[0])
print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)
print("DGL:", dgl.__version__)
'''


verification = subprocess.run(
    [
        LEGACY_PYTHON,
        "-c",
        verification_code
    ],
    capture_output=True,
    text=True
)


print(
    "\n===== LEGACY ENVIRONMENT ====="
)

print(
    verification.stdout
)


if verification.returncode != 0:

    print(
        verification.stderr
    )

    raise RuntimeError(
        "Legacy environment verification failed."
    )


# ============================================================
# 9. TASK 5 OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = os.path.join(
    WORK_DIR,
    "tsocial_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


TASK5_JSON = os.path.join(
    OUTPUT_DIR,
    "tsocial_task5_global_heterophily.json"
)

DEGREE_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_undirected_degree.npy"
)

DIFF_DEGREE_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_different_label_degree.npy"
)

LABELS_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_labels.npy"
)


# ============================================================
# 10. CREATE LEGACY TASK 5 SCRIPT
# ============================================================

TASK5_SCRIPT = os.path.join(
    WORK_DIR,
    "tsocial_task5_legacy.py"
)


legacy_code = r'''
import os
import json
import numpy as np
import torch
import dgl

from dgl.data.utils import load_graphs


SOURCE = (
    "/kaggle/working/"
    "tsocial_official_source/"
    "tsocial"
)

OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


print(
    "===== T-SOCIAL TASK 5 LEGACY ANALYSIS ====="
)


# ============================================================
# A. LOAD OFFICIAL GRAPH
# ============================================================

print(
    "\nLoading official T-Social graph..."
)


graphs, metadata = load_graphs(
    SOURCE
)


if len(graphs) != 1:

    raise RuntimeError(
        f"Expected one graph, found {len(graphs)}."
    )


g = graphs[0]


N = int(
    g.num_nodes()
)

E = int(
    g.num_edges()
)


print(
    "\nPASS: Official graph loaded."
)

print(
    "Nodes:",
    f"{N:,}"
)

print(
    "Stored DGL edge entries:",
    f"{E:,}"
)


# ============================================================
# B. VERIFY EXPECTED GRAPH SCALE
# ============================================================

if N != 5_781_065:

    raise RuntimeError(
        f"Unexpected node count: {N:,}"
    )


if E != 146_211_016:

    raise RuntimeError(
        f"Unexpected stored-edge count: {E:,}"
    )


features = g.ndata[
    "feature"
]


if features.ndim != 2:

    raise RuntimeError(
        "Unexpected feature representation."
    )


if int(
    features.shape[1]
) != 10:

    raise RuntimeError(
        "Unexpected feature dimension."
    )


print(
    "PASS: Graph scale verified."
)


# ============================================================
# C. LABELS
# ============================================================

labels = (
    g.ndata["label"]
    .long()
    .squeeze(-1)
)


if labels.ndim != 1:

    raise RuntimeError(
        "Unexpected label shape."
    )


if len(labels) != N:

    raise RuntimeError(
        "Label count does not match nodes."
    )


unique_labels = torch.unique(
    labels
)


if set(
    unique_labels.tolist()
) != {0, 1}:

    raise RuntimeError(
        "Expected labels {0,1}."
    )


print(
    "\nPASS: Binary labels verified."
)


# Save labels for Task 6

labels_np = (
    labels
    .cpu()
    .numpy()
    .astype(
        np.int8,
        copy=False
    )
)


np.save(
    os.path.join(
        OUTPUT_DIR,
        "tsocial_labels.npy"
    ),
    labels_np
)


print(
    "PASS: Labels saved for Task 6."
)


# ============================================================
# D. CHECK PARALLEL DIRECTED EDGES
# ============================================================

print(
    "\nChecking whether DGL reports "
    "parallel directed edges..."
)


multigraph_attribute = (
    g.is_multigraph
)


if callable(
    multigraph_attribute
):

    is_multigraph = bool(
        multigraph_attribute()
    )

else:

    is_multigraph = bool(
        multigraph_attribute
    )


print(
    "DGL is_multigraph:",
    is_multigraph
)


if is_multigraph:

    raise RuntimeError(
        "Parallel directed edges detected.\n"
        "Stop before applying the one-edge-per-"
        "undirected-pair convention."
    )


print(
    "PASS: No parallel directed edges detected."
)


# ============================================================
# E. ARRAYS FOR LOCAL HETEROPHILY
# ============================================================

# degree[i]:
# number of undirected non-self neighbours
#
# different_degree[i]:
# number of those neighbours having a different label

degree = torch.zeros(
    N,
    dtype=torch.int64
)

different_degree = torch.zeros(
    N,
    dtype=torch.int64
)


# ============================================================
# F. STRUCTURAL COUNTERS
# ============================================================

self_loops = 0

upper_edges = 0

lower_edges = 0

missing_reverse_edges = 0

heterophilic_unique_edges = 0


CHUNK_SIZE = (
    1_000_000
)

num_chunks = (
    E
    + CHUNK_SIZE
    - 1
) // CHUNK_SIZE


print(
    "\n===== STRUCTURAL EDGE SCAN ====="
)

print(
    "Stored entries:",
    f"{E:,}"
)

print(
    "Chunk size:",
    f"{CHUNK_SIZE:,}"
)

print(
    "Number of chunks:",
    num_chunks
)


# ============================================================
# G. SCAN ALL STORED EDGES
# ============================================================

for chunk_number, start in enumerate(
    range(
        0,
        E,
        CHUNK_SIZE
    ),
    start=1
):

    end = min(
        start + CHUNK_SIZE,
        E
    )


    eids = torch.arange(
        start,
        end,
        dtype=torch.int64
    )


    src, dst = g.find_edges(
        eids
    )


    # --------------------------------------------------------
    # Self-loops
    # --------------------------------------------------------

    loop_mask = (
        src == dst
    )


    self_loops += int(
        loop_mask
        .sum()
        .item()
    )


    # --------------------------------------------------------
    # Canonical orientation
    # --------------------------------------------------------

    upper_mask = (
        src < dst
    )

    lower_mask = (
        src > dst
    )


    upper_edges += int(
        upper_mask
        .sum()
        .item()
    )


    lower_edges += int(
        lower_mask
        .sum()
        .item()
    )


    # --------------------------------------------------------
    # Reciprocity check
    #
    # For every stored u -> v,
    # verify that v -> u exists.
    # --------------------------------------------------------

    reverse_exists = g.has_edges_between(
        dst,
        src
    )


    missing_reverse_edges += int(
        (~reverse_exists)
        .sum()
        .item()
    )


    # --------------------------------------------------------
    # Use src < dst as one canonical copy
    # of each candidate undirected edge.
    # --------------------------------------------------------

    if bool(
        upper_mask.any()
    ):

        u = src[
            upper_mask
        ]

        v = dst[
            upper_mask
        ]


        label_diff = (
            labels[u]
            !=
            labels[v]
        )


        heterophilic_unique_edges += int(
            label_diff
            .sum()
            .item()
        )


        # ----------------------------------------------------
        # Degree accumulation
        # ----------------------------------------------------

        ones = torch.ones(
            len(u),
            dtype=torch.int64
        )


        degree.scatter_add_(
            0,
            u,
            ones
        )


        degree.scatter_add_(
            0,
            v,
            ones
        )


        # ----------------------------------------------------
        # Different-label neighbour counts
        # ----------------------------------------------------

        if bool(
            label_diff.any()
        ):

            u_diff = u[
                label_diff
            ]

            v_diff = v[
                label_diff
            ]


            diff_ones = torch.ones(
                len(u_diff),
                dtype=torch.int64
            )


            different_degree.scatter_add_(
                0,
                u_diff,
                diff_ones
            )


            different_degree.scatter_add_(
                0,
                v_diff,
                diff_ones
            )


    if (
        chunk_number % 10 == 0
        or
        chunk_number == num_chunks
    ):

        print(
            f"Chunk "
            f"{chunk_number}/"
            f"{num_chunks} complete"
        )


# ============================================================
# H. STRUCTURAL VERIFICATION
# ============================================================

print(
    "\n===== STRUCTURAL VERIFICATION ====="
)


print(
    "Raw stored DGL edge entries:",
    f"{E:,}"
)

print(
    "Self-loop entries:",
    f"{self_loops:,}"
)

print(
    "src < dst entries:",
    f"{upper_edges:,}"
)

print(
    "src > dst entries:",
    f"{lower_edges:,}"
)

print(
    "Stored entries missing reverse counterpart:",
    f"{missing_reverse_edges:,}"
)


# ------------------------------------------------------------
# Every stored edge must belong to exactly one category:
# src<dst, src>dst or src==dst
# ------------------------------------------------------------

partition_total = (
    upper_edges
    + lower_edges
    + self_loops
)


if partition_total != E:

    raise RuntimeError(
        "Edge partition failed.\n"
        f"Partition total = {partition_total:,}\n"
        f"Stored total    = {E:,}"
    )


print(
    "\nPASS: Upper/lower/self-loop partition "
    "matches total stored entries."
)


# ------------------------------------------------------------
# Reciprocity
# ------------------------------------------------------------

if missing_reverse_edges != 0:

    raise RuntimeError(
        "Graph is not completely reciprocal.\n"
        "Do NOT divide stored-edge count by two."
    )


print(
    "PASS: Every stored edge has a reverse counterpart."
)


# ------------------------------------------------------------
# Direction equality
# ------------------------------------------------------------

if upper_edges != lower_edges:

    raise RuntimeError(
        "Upper/lower directional counts differ."
    )


print(
    "PASS: Upper and lower directional counts agree."
)


# ============================================================
# I. FINAL COMPARABLE UNDIRECTED EDGE COUNT
# ============================================================

# Because:
#
# 1. no parallel directed edges exist,
# 2. every non-self edge has a reverse,
# 3. src<dst selects exactly one orientation,
#
# upper_edges is the unique undirected non-self count.

unique_undirected_edges = int(
    upper_edges
)


eligible_undirected_edges = int(
    unique_undirected_edges
)


print(
    "\n===== UNDIRECTED STRUCTURE ====="
)

print(
    "Unique undirected non-self edges:",
    f"{unique_undirected_edges:,}"
)

print(
    "Eligible labelled undirected edges:",
    f"{eligible_undirected_edges:,}"
)


# ============================================================
# J. GLOBAL HETEROPHILY
# ============================================================

homophilic_unique_edges = int(
    eligible_undirected_edges
    -
    heterophilic_unique_edges
)


if eligible_undirected_edges == 0:

    raise RuntimeError(
        "No eligible undirected edges."
    )


global_heterophily = (
    heterophilic_unique_edges
    /
    eligible_undirected_edges
)


print(
    "\n===== GLOBAL HETEROPHILY ====="
)


print(
    "Homophilic undirected edges:",
    f"{homophilic_unique_edges:,}"
)

print(
    "Heterophilic undirected edges:",
    f"{heterophilic_unique_edges:,}"
)

print(
    "Eligible undirected edges:",
    f"{eligible_undirected_edges:,}"
)

print(
    "Global heterophily:",
    f"{global_heterophily:.9f}"
)

print(
    "Global heterophily (%):",
    f"{global_heterophily * 100:.6f}%"
)


# ============================================================
# K. HANDSHAKE CHECK
# ============================================================

degree_sum = int(
    degree
    .sum()
    .item()
)

expected_degree_sum = (
    2
    *
    unique_undirected_edges
)


print(
    "\n===== DEGREE VALIDATION ====="
)

print(
    "Degree sum:",
    f"{degree_sum:,}"
)

print(
    "Expected 2|E|:",
    f"{expected_degree_sum:,}"
)


if degree_sum != expected_degree_sum:

    raise RuntimeError(
        "Degree handshake identity failed."
    )


print(
    "PASS: Degree handshake identity verified."
)


different_degree_sum = int(
    different_degree
    .sum()
    .item()
)

expected_different_sum = (
    2
    *
    heterophilic_unique_edges
)


print(
    "Different-label degree sum:",
    f"{different_degree_sum:,}"
)

print(
    "Expected 2 × heterophilic edges:",
    f"{expected_different_sum:,}"
)


if (
    different_degree_sum
    !=
    expected_different_sum
):

    raise RuntimeError(
        "Different-label degree identity failed."
    )


print(
    "PASS: Heterophilic-degree identity verified."
)


# ============================================================
# L. SAVE TASK 6 INPUT ARRAYS
# ============================================================

np.save(
    os.path.join(
        OUTPUT_DIR,
        "tsocial_undirected_degree.npy"
    ),
    degree.numpy()
)


np.save(
    os.path.join(
        OUTPUT_DIR,
        "tsocial_different_label_degree.npy"
    ),
    different_degree.numpy()
)


print(
    "\nPASS: Degree arrays saved for Task 6."
)


# ============================================================
# M. SAVE TASK 5 RESULTS
# ============================================================

results = {

    "dataset":
        "T-Social",

    "nodes":
        N,

    "raw_stored_dgl_edge_entries":
        E,

    "is_multigraph":
        is_multigraph,

    "self_loop_entries":
        int(self_loops),

    "upper_direction_entries":
        int(upper_edges),

    "lower_direction_entries":
        int(lower_edges),

    "missing_reverse_entries":
        int(missing_reverse_edges),

    "fully_reciprocal":
        bool(
            missing_reverse_edges == 0
        ),

    "unique_undirected_nonself_edges":
        int(
            unique_undirected_edges
        ),

    "eligible_labelled_undirected_edges":
        int(
            eligible_undirected_edges
        ),

    "homophilic_undirected_edges":
        int(
            homophilic_unique_edges
        ),

    "heterophilic_undirected_edges":
        int(
            heterophilic_unique_edges
        ),

    "global_heterophily":
        float(
            global_heterophily
        ),

    "global_heterophily_percent":
        float(
            global_heterophily
            * 100.0
        ),

    "method":
        (
            "One src<dst edge per undirected "
            "non-self pair after verifying "
            "no parallel directed edges and "
            "complete reciprocity."
        ),

    "self_loop_policy":
        "Excluded",

    "label_policy":
        (
            "All nodes have binary labels; "
            "all non-self structural edges eligible."
        ),

    "local_h_std_convention":
        (
            "Population standard deviation "
            "(ddof=0)"
        )
}


with open(
    os.path.join(
        OUTPUT_DIR,
        "tsocial_task5_global_heterophily.json"
    ),
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )


print(
    "\nPASS: Task 5 results saved."
)


# ============================================================
# N. FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 5 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nValues to record:"
)

print(
    "Raw stored DGL entries =",
    f"{E:,}"
)

print(
    "DGL multigraph =",
    is_multigraph
)

print(
    "Self-loop entries =",
    f"{self_loops:,}"
)

print(
    "Missing reverse entries =",
    f"{missing_reverse_edges:,}"
)

print(
    "Unique undirected non-self edges =",
    f"{unique_undirected_edges:,}"
)

print(
    "Homophilic undirected edges =",
    f"{homophilic_unique_edges:,}"
)

print(
    "Heterophilic undirected edges =",
    f"{heterophilic_unique_edges:,}"
)

print(
    "Global heterophily =",
    f"{global_heterophily:.9f}"
)
'''


with open(
    TASK5_SCRIPT,
    "w"
) as f:

    f.write(
        legacy_code
    )


print(
    "\nLegacy Task 5 analysis script created."
)


# ============================================================
# 11. RUN TASK 5
# ============================================================

print(
    "\nStarting structural verification "
    "and global heterophily calculation..."
)

print(
    "The calculation scans all "
    "146,211,016 stored DGL edge entries."
)

print(
    "Do not interrupt the Kaggle session."
)


process = subprocess.Popen(
    [
        LEGACY_PYTHON,
        TASK5_SCRIPT
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


for line in process.stdout:

    print(
        line,
        end=""
    )


return_code = process.wait()


if return_code != 0:

    raise RuntimeError(
        "T-Social Task 5 failed with "
        f"return code {return_code}."
    )


# ============================================================
# 12. VERIFY TASK 5 OUTPUTS
# ============================================================

if not os.path.isfile(
    TASK5_JSON
):

    raise FileNotFoundError(
        "Task 5 JSON output was not created."
    )


if not os.path.isfile(
    DEGREE_PATH
):

    raise FileNotFoundError(
        "Undirected degree array missing."
    )


if not os.path.isfile(
    DIFF_DEGREE_PATH
):

    raise FileNotFoundError(
        "Different-label degree array missing."
    )


if not os.path.isfile(
    LABELS_PATH
):

    raise FileNotFoundError(
        "Task 6 label array missing."
    )


# ============================================================
# 13. LOAD FINAL RESULT
# ============================================================

with open(
    TASK5_JSON,
    "r"
) as f:

    task5_results = json.load(
        f
    )


print(
    "\n===== CURRENT NOTEBOOK CHECK ====="
)


print(
    "Raw stored entries:",
    f"{task5_results['raw_stored_dgl_edge_entries']:,}"
)

print(
    "Self loops:",
    f"{task5_results['self_loop_entries']:,}"
)

print(
    "Missing reverse entries:",
    f"{task5_results['missing_reverse_entries']:,}"
)

print(
    "Unique undirected edges:",
    f"{task5_results['unique_undirected_nonself_edges']:,}"
)

print(
    "Global heterophily:",
    f"{task5_results['global_heterophily']:.9f}"
)


print(
    "\nPASS: Task 5 completed successfully."
)

print(
    "PASS: Task 6 degree and label arrays are ready."
)


print(
    "\nIMPORTANT:"
)

print(
    "Do NOT restart the Kaggle session "
    "before completing Task 6."
)

===== T-SOCIAL TASK 5 =====

Official extracted graph is not currently available.
Recovering the authors' official T-Social release...

Installing/updating gdown...

Google Drive source:
https://drive.google.com/uc?id=10nY_IwxT32KdfpcTqoy8wgckXK2xayXd


Downloading...
From (original): https://drive.google.com/uc?id=10nY_IwxT32KdfpcTqoy8wgckXK2xayXd
From (redirected): https://drive.google.com/uc?id=10nY_IwxT32KdfpcTqoy8wgckXK2xayXd&confirm=t&uuid=88d89cd1-a65b-4a5f-8731-0963373b6d83
To: /kaggle/working/tsocial_official_source/tsocial.zip
100%|██████████| 744M/744M [00:06<00:00, 108MB/s] 



Official ZIP size: 744,239,223 bytes

Checking ZIP integrity...
Archive contents: ['tsocial']

Extracting official T-Social graph...
PASS: Official archive extracted.

Official graph:
/kaggle/working/tsocial_official_source/tsocial
Graph size: 4,110,300,257 bytes

Calculating official graph SHA256...
SHA256:
8d577114cff12f7de35eda2974f17825ef9febe20c661e95e8d21072a6dfc8d0

PASS: Official source identity verified.

Legacy environment is missing.
Recreating Python 3.9 / PyTorch 1.9 / DGL 0.8.1...

bin/micromamba

Creating Python 3.9 environment...
Fetch Shard Index for conda-forge/linux-64                                                      ⧖ Starting
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.2 sec)
Fetch Shard Index for conda-forge/noarch                                                        ⧖ Starting
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.2 sec)
Fetching and Parsin

warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.


Linking libzlib-1.3.2-h25fd6f3_3
Linking libgomp-16.2.0-he0feb66_4
Linking zstd-1.5.7-hb78ec9c_7
Linking _openmp_mutex-4.5-20_gnu
Linking ld_impl_linux-64-2.46.1-default_hbd61a6d_102
Linking libgcc-16.2.0-ha9f2e26_4
Linking libstdcxx-16.2.0-h934c35e_4
Linking bzip2-1.0.8-hda65f42_10
Linking libnsl-2.0.1-hb9d3cd8_1
Linking libexpat-2.8.1-hecca717_1
Linking libuuid-2.42.3-hcfc3c73_0
Linking libffi-3.7.0-h81df57d_1
Linking libxcrypt-4.4.38-h280c20c_0
Linking liblzma-5.8.3-hb03c661_1
Linking ncurses-6.6-hdb14827_1
Linking tk-8.6.13-noxft_h1df4ec4_4
Linking icu-78.3-py310h44b86e0_2
Linking readline-8.3-hd6e31c0_1
Linking libsqlite-3.53.4-h13e7031_1
Linking tzdata-2026c-h151e31d_0
Linking ca-certificates-2026.7.22-hbd8a1cb_0
Linking openssl-3.6.4-h781a0a9_0
Linking python-3.9.23-hc30ae73_0_cpython
Linking wheel-0.45.1-pyhd8ed1ab_1
Linking setuptools-80.9.0-pyhff2d567_0
Linking pip-25.2-pyh8b19718_0

Transaction finished


To activate this environment, use:

    micromamba activate /kaggle/wo

## Task 6 — Calculate local heterophily

Global heterophily describes the proportion of all graph edges that connect
nodes with different labels. Local heterophily instead measures this proportion
separately for each node.

For node \(i\), local heterophily is defined as:

\[
H_i =
\frac{\text{number of neighbours of } i \text{ with a different label}}
{\text{number of labelled neighbours of } i}
\]

The graph structure used here is the verified simple undirected T-Social graph
from Task 5:

- 73,105,508 unique undirected non-self edges
- no self-loops
- no parallel directed edges
- complete reciprocal storage in the original DGL graph

All T-Social nodes have binary labels.

Nodes with degree zero have no neighbours, so local heterophily is undefined
for them. These nodes are excluded from the local-heterophily summary
statistics and are reported separately as isolates.

For comparability with the YelpChi analysis, the following statistics are
calculated across non-isolated nodes:

- mean
- median
- population standard deviation (`ddof=0`)
- first quartile (Q1)
- third quartile (Q3)
- minimum
- maximum
- mean for normal nodes
- mean for anomaly nodes
- number of isolates

A weighted consistency check is also performed:

\[
\frac{\sum_i \text{different-label degree}_i}
{\sum_i \text{degree}_i}
=
H_{\text{global}}
\]

This weighted local quantity must equal the global heterophily calculated in
Task 5.

In [8]:
# CELL 6 — T-SOCIAL TASK 6
# LOCAL HETEROPHILY
#
# Uses the degree, different-label degree and label arrays
# created in Task 5.
#
# IMPORTANT:
# - isolates (degree = 0) are excluded from local-H summaries
# - population standard deviation is used (ddof=0)
# - local H is computed on the verified simple undirected graph

import os
import json
import numpy as np


print("===== T-SOCIAL TASK 6 =====")


# ============================================================
# 1. INPUT PATHS FROM TASK 5
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)


TASK5_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task5_global_heterophily.json"
)

DEGREE_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_undirected_degree.npy"
)

DIFF_DEGREE_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_different_label_degree.npy"
)

LABELS_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_labels.npy"
)


required_files = [
    TASK5_PATH,
    DEGREE_PATH,
    DIFF_DEGREE_PATH,
    LABELS_PATH
]


for path in required_files:

    if not os.path.isfile(path):

        raise FileNotFoundError(
            "Required Task 5 file is missing:\n"
            f"{path}\n\n"
            "Do not restart the Kaggle session "
            "between Tasks 5 and 6."
        )


print(
    "\nPASS: All Task 5 inputs found."
)


# ============================================================
# 2. LOAD TASK 5 RESULT
# ============================================================

with open(
    TASK5_PATH,
    "r"
) as f:

    task5_results = json.load(
        f
    )


num_nodes = int(
    task5_results[
        "nodes"
    ]
)

unique_undirected_edges = int(
    task5_results[
        "unique_undirected_nonself_edges"
    ]
)

global_heterophily = float(
    task5_results[
        "global_heterophily"
    ]
)


print(
    "\n===== VERIFIED TASK 5 STRUCTURE ====="
)

print(
    "Nodes:",
    f"{num_nodes:,}"
)

print(
    "Unique undirected edges:",
    f"{unique_undirected_edges:,}"
)

print(
    "Global heterophily:",
    f"{global_heterophily:.9f}"
)


# ============================================================
# 3. LOAD ARRAYS
#
# mmap_mode avoids unnecessary duplication of the two
# int64 degree arrays in RAM.
# ============================================================

degree = np.load(
    DEGREE_PATH,
    mmap_mode="r"
)

different_degree = np.load(
    DIFF_DEGREE_PATH,
    mmap_mode="r"
)

labels = np.load(
    LABELS_PATH,
    mmap_mode="r"
)


print(
    "\n===== ARRAY INFORMATION ====="
)

print(
    "Degree:",
    degree.shape,
    degree.dtype
)

print(
    "Different-label degree:",
    different_degree.shape,
    different_degree.dtype
)

print(
    "Labels:",
    labels.shape,
    labels.dtype
)


# ============================================================
# 4. BASIC INTEGRITY CHECKS
# ============================================================

assert (
    len(degree)
    == num_nodes
)

assert (
    len(different_degree)
    == num_nodes
)

assert (
    len(labels)
    == num_nodes
)


assert np.all(
    degree >= 0
)

assert np.all(
    different_degree >= 0
)

assert np.all(
    different_degree <= degree
)


unique_labels = np.unique(
    labels
)


assert set(
    unique_labels.tolist()
) == {0, 1}


print(
    "\nPASS: Array lengths match node count."
)

print(
    "PASS: Different-label degree never exceeds degree."
)

print(
    "PASS: Binary labels verified."
)


# ============================================================
# 5. DEGREE HANDSHAKE RECHECK
# ============================================================

degree_sum = int(
    np.sum(
        degree,
        dtype=np.int64
    )
)

expected_degree_sum = (
    2
    * unique_undirected_edges
)


print(
    "\n===== DEGREE RECHECK ====="
)

print(
    "Degree sum:",
    f"{degree_sum:,}"
)

print(
    "Expected 2|E|:",
    f"{expected_degree_sum:,}"
)


assert (
    degree_sum
    == expected_degree_sum
)


different_degree_sum = int(
    np.sum(
        different_degree,
        dtype=np.int64
    )
)


print(
    "Different-label degree sum:",
    f"{different_degree_sum:,}"
)


print(
    "\nPASS: Degree handshake identity still holds."
)


# ============================================================
# 6. IDENTIFY ISOLATES
# ============================================================

non_isolate_mask = (
    degree > 0
)

isolate_mask = (
    degree == 0
)


non_isolate_count = int(
    np.count_nonzero(
        non_isolate_mask
    )
)

isolate_count = int(
    np.count_nonzero(
        isolate_mask
    )
)


assert (
    non_isolate_count
    + isolate_count
    == num_nodes
)


normal_isolates = int(
    np.count_nonzero(
        isolate_mask
        &
        (labels == 0)
    )
)

anomaly_isolates = int(
    np.count_nonzero(
        isolate_mask
        &
        (labels == 1)
    )
)


print(
    "\n===== ISOLATES ====="
)

print(
    "Non-isolated nodes:",
    f"{non_isolate_count:,}"
)

print(
    "Isolated nodes:",
    f"{isolate_count:,}"
)

print(
    "Normal isolates:",
    f"{normal_isolates:,}"
)

print(
    "Anomaly isolates:",
    f"{anomaly_isolates:,}"
)


# ============================================================
# 7. CALCULATE LOCAL HETEROPHILY
#
# Only degree > 0 nodes receive a defined local H value.
# Use float64 for summary accuracy.
# ============================================================

print(
    "\nCalculating node-level local heterophily..."
)


degree_nonzero = np.asarray(
    degree[
        non_isolate_mask
    ],
    dtype=np.float64
)


different_nonzero = np.asarray(
    different_degree[
        non_isolate_mask
    ],
    dtype=np.float64
)


local_h_nonisolates = (
    different_nonzero
    /
    degree_nonzero
)


assert np.all(
    local_h_nonisolates >= 0.0
)

assert np.all(
    local_h_nonisolates <= 1.0
)


print(
    "PASS: All defined local-H values are "
    "between 0 and 1."
)


# ============================================================
# 8. OVERALL LOCAL-H SUMMARY
# ============================================================

local_h_mean = float(
    np.mean(
        local_h_nonisolates
    )
)

local_h_median = float(
    np.median(
        local_h_nonisolates
    )
)

# Population SD — same convention as YelpChi
local_h_std = float(
    np.std(
        local_h_nonisolates,
        ddof=0
    )
)


q1, q3 = np.percentile(
    local_h_nonisolates,
    [
        25,
        75
    ]
)


local_h_q1 = float(q1)

local_h_q3 = float(q3)


local_h_min = float(
    np.min(
        local_h_nonisolates
    )
)

local_h_max = float(
    np.max(
        local_h_nonisolates
    )
)


# ============================================================
# 9. NORMAL VS ANOMALY LOCAL H
# ============================================================

labels_nonisolates = np.asarray(
    labels[
        non_isolate_mask
    ]
)


normal_local_mask = (
    labels_nonisolates == 0
)

anomaly_local_mask = (
    labels_nonisolates == 1
)


normal_nonisolate_count = int(
    np.count_nonzero(
        normal_local_mask
    )
)

anomaly_nonisolate_count = int(
    np.count_nonzero(
        anomaly_local_mask
    )
)


normal_local_h_mean = float(
    np.mean(
        local_h_nonisolates[
            normal_local_mask
        ]
    )
)


anomaly_local_h_mean = float(
    np.mean(
        local_h_nonisolates[
            anomaly_local_mask
        ]
    )
)


# ============================================================
# 10. WEIGHTED LOCAL-H / GLOBAL-H IDENTITY
#
# This must equal:
#
#   heterophilic edges / total edges
#
# because each undirected edge contributes twice to degree.
# ============================================================

weighted_local_h = (
    different_degree_sum
    /
    degree_sum
)


difference_from_global = abs(
    weighted_local_h
    -
    global_heterophily
)


print(
    "\n===== GLOBAL / LOCAL CONSISTENCY ====="
)

print(
    "Global heterophily from Task 5:",
    f"{global_heterophily:.12f}"
)

print(
    "Degree-weighted local heterophily:",
    f"{weighted_local_h:.12f}"
)

print(
    "Absolute difference:",
    f"{difference_from_global:.16f}"
)


assert (
    difference_from_global
    < 1e-12
), (
    "Weighted local heterophily does not "
    "match global heterophily."
)


print(
    "PASS: Weighted local-H identity "
    "matches global heterophily."
)


# ============================================================
# 11. SAVE FULL NODE-LEVEL LOCAL H ARRAY
#
# NaN = isolate / undefined local heterophily.
#
# float32 is sufficient for the saved node-level values.
# Summary statistics above were calculated using float64.
# ============================================================

LOCAL_H_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_local_heterophily.npy"
)


local_h_all = np.lib.format.open_memmap(
    LOCAL_H_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(num_nodes,)
)


local_h_all[:] = np.nan


local_h_all[
    non_isolate_mask
] = local_h_nonisolates.astype(
    np.float32,
    copy=False
)


local_h_all.flush()

del local_h_all


print(
    "\nPASS: Full node-level local-H array saved."
)

print(
    LOCAL_H_PATH
)


# ============================================================
# 12. DISPLAY SUMMARY
# ============================================================

print(
    "\n===== LOCAL HETEROPHILY SUMMARY ====="
)


print(
    "Nodes with defined local H:",
    f"{non_isolate_count:,}"
)

print(
    "Isolates excluded:",
    f"{isolate_count:,}"
)


print(
    "\nMean:",
    f"{local_h_mean:.9f}"
)

print(
    "Median:",
    f"{local_h_median:.9f}"
)

print(
    "Population SD:",
    f"{local_h_std:.9f}"
)

print(
    "Q1:",
    f"{local_h_q1:.9f}"
)

print(
    "Q3:",
    f"{local_h_q3:.9f}"
)

print(
    "Minimum:",
    f"{local_h_min:.9f}"
)

print(
    "Maximum:",
    f"{local_h_max:.9f}"
)


print(
    "\n===== CLASS-SPECIFIC LOCAL H ====="
)

print(
    "Normal non-isolated nodes:",
    f"{normal_nonisolate_count:,}"
)

print(
    "Normal-node mean local H:",
    f"{normal_local_h_mean:.9f}"
)

print(
    "Anomaly non-isolated nodes:",
    f"{anomaly_nonisolate_count:,}"
)

print(
    "Anomaly-node mean local H:",
    f"{anomaly_local_h_mean:.9f}"
)


# ============================================================
# 13. SAVE TASK 6 SUMMARY
# ============================================================

task6_results = {

    "dataset":
        "T-Social",

    "local_heterophily_definition":
        (
            "Different-label neighbours divided by "
            "total labelled neighbours"
        ),

    "graph_edge_convention":
        (
            "Unique undirected non-self edges"
        ),

    "isolates_policy":
        (
            "Degree-zero nodes excluded from "
            "local-H summary statistics"
        ),

    "std_convention":
        (
            "Population standard deviation (ddof=0)"
        ),

    "total_nodes":
        int(num_nodes),

    "nodes_with_defined_local_h":
        int(non_isolate_count),

    "isolates":
        int(isolate_count),

    "normal_isolates":
        int(normal_isolates),

    "anomaly_isolates":
        int(anomaly_isolates),

    "mean":
        local_h_mean,

    "median":
        local_h_median,

    "population_std":
        local_h_std,

    "q1":
        local_h_q1,

    "q3":
        local_h_q3,

    "minimum":
        local_h_min,

    "maximum":
        local_h_max,

    "normal_nonisolated_nodes":
        int(normal_nonisolate_count),

    "anomaly_nonisolated_nodes":
        int(anomaly_nonisolate_count),

    "normal_mean_local_h":
        normal_local_h_mean,

    "anomaly_mean_local_h":
        anomaly_local_h_mean,

    "global_heterophily":
        float(
            global_heterophily
        ),

    "degree_weighted_local_heterophily":
        float(
            weighted_local_h
        ),

    "weighted_identity_absolute_difference":
        float(
            difference_from_global
        )
}


TASK6_JSON = os.path.join(
    OUTPUT_DIR,
    "tsocial_task6_local_heterophily.json"
)


with open(
    TASK6_JSON,
    "w"
) as f:

    json.dump(
        task6_results,
        f,
        indent=4
    )


print(
    "\nPASS: Task 6 summary saved:"
)

print(
    TASK6_JSON
)


# ============================================================
# 14. FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 6 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nValues to record:"
)

print(
    "Local H mean =",
    f"{local_h_mean:.9f}"
)

print(
    "Local H median =",
    f"{local_h_median:.9f}"
)

print(
    "Local H population SD =",
    f"{local_h_std:.9f}"
)

print(
    "Local H Q1 =",
    f"{local_h_q1:.9f}"
)

print(
    "Local H Q3 =",
    f"{local_h_q3:.9f}"
)

print(
    "Local H min =",
    f"{local_h_min:.9f}"
)

print(
    "Local H max =",
    f"{local_h_max:.9f}"
)

print(
    "Normal-node mean local H =",
    f"{normal_local_h_mean:.9f}"
)

print(
    "Anomaly-node mean local H =",
    f"{anomaly_local_h_mean:.9f}"
)

print(
    "Isolates =",
    f"{isolate_count:,}"
)

print(
    "Weighted local H =",
    f"{weighted_local_h:.9f}"
)

print(
    "Global H =",
    f"{global_heterophily:.9f}"
)

===== T-SOCIAL TASK 6 =====

PASS: All Task 5 inputs found.

===== VERIFIED TASK 5 STRUCTURE =====
Nodes: 5,781,065
Unique undirected edges: 73,105,508
Global heterophily: 0.376095095

===== ARRAY INFORMATION =====
Degree: (5781065,) int64
Different-label degree: (5781065,) int64
Labels: (5781065,) int8

PASS: Array lengths match node count.
PASS: Different-label degree never exceeds degree.
PASS: Binary labels verified.

===== DEGREE RECHECK =====
Degree sum: 146,211,016
Expected 2|E|: 146,211,016
Different-label degree sum: 54,989,246

PASS: Degree handshake identity still holds.

===== ISOLATES =====
Non-isolated nodes: 5,781,065
Isolated nodes: 0
Normal isolates: 0
Anomaly isolates: 0

Calculating node-level local heterophily...
PASS: All defined local-H values are between 0 and 1.

===== GLOBAL / LOCAL CONSISTENCY =====
Global heterophily from Task 5: 0.376095095324
Degree-weighted local heterophily: 0.376095095324
Absolute difference: 0.0000000000000000
PASS: Weighted local-H ide

## Task 7 — Reproduce the original/reference data split

This step reproduces the train-validation-test split used by the official
BWGNN implementation for T-Social.

The official BWGNN code first performs a stratified random split with:

- training proportion = 0.40
- random state = 2
- shuffle = True

The remaining 60% of nodes are then divided using:

- test_size = 0.67
- random state = 2
- shuffle = True
- stratification by class label

Therefore, the resulting split is approximately:

- 40.0% training
- 19.8% validation
- 40.2% testing

This is a random stratified node split, not a temporal split.

The split is reproduced here using the actual T-Social labels. The resulting
indices and class counts are saved so that later reproduction experiments can
use exactly the same split.

This original/reference split is kept separate from any future unified split
that may be created for fair comparison across all benchmark datasets.

In [9]:
# CELL 7 — T-SOCIAL TASK 7
# ORIGINAL / REFERENCE TRAIN-VALIDATION-TEST SPLIT

import os
import json
import numpy as np

from sklearn.model_selection import train_test_split


print("===== T-SOCIAL TASK 7 =====")


# ============================================================
# 1. INPUT
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

LABELS_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_labels.npy"
)


if not os.path.isfile(LABELS_PATH):

    raise FileNotFoundError(
        "T-Social labels from Task 5 were not found:\n"
        f"{LABELS_PATH}"
    )


labels = np.load(
    LABELS_PATH,
    mmap_mode="r"
)


num_nodes = int(
    len(labels)
)


print(
    "\nNodes:",
    f"{num_nodes:,}"
)

print(
    "Labels shape:",
    labels.shape
)

print(
    "Labels dtype:",
    labels.dtype
)


# ============================================================
# 2. VERIFY LABELS
# ============================================================

unique_labels, full_counts = np.unique(
    labels,
    return_counts=True
)


assert set(
    unique_labels.tolist()
) == {0, 1}


normal_total = int(
    full_counts[
        np.where(
            unique_labels == 0
        )[0][0]
    ]
)

anomaly_total = int(
    full_counts[
        np.where(
            unique_labels == 1
        )[0][0]
    ]
)


assert (
    normal_total == 5_606_785
)

assert (
    anomaly_total == 174_280
)


print(
    "\nPASS: Actual T-Social labels verified."
)


# ============================================================
# 3. NODE INDICES
# ============================================================

# int32 is sufficient because the maximum node ID
# is below 5.8 million.

indices = np.arange(
    num_nodes,
    dtype=np.int32
)


# ============================================================
# 4. OFFICIAL BWGNN FIRST SPLIT
#
# Official main.py:
#
# train_test_split(
#     index,
#     labels[index],
#     stratify=labels[index],
#     train_size=args.train_ratio,
#     random_state=2,
#     shuffle=True
# )
#
# T-Social is run with:
# --train_ratio 0.4
# ============================================================

idx_train, idx_rest, y_train, y_rest = (
    train_test_split(
        indices,
        np.asarray(labels),
        stratify=np.asarray(labels),
        train_size=0.4,
        random_state=2,
        shuffle=True
    )
)


print(
    "\nPASS: First 40% stratified split created."
)


# ============================================================
# 5. OFFICIAL BWGNN SECOND SPLIT
#
# Official main.py:
#
# train_test_split(
#     idx_rest,
#     y_rest,
#     stratify=y_rest,
#     test_size=0.67,
#     random_state=2,
#     shuffle=True
# )
# ============================================================

idx_valid, idx_test, y_valid, y_test = (
    train_test_split(
        idx_rest,
        y_rest,
        stratify=y_rest,
        test_size=0.67,
        random_state=2,
        shuffle=True
    )
)


print(
    "PASS: Validation/test split created."
)


# ============================================================
# 6. SPLIT SIZES
# ============================================================

train_count = int(
    len(idx_train)
)

valid_count = int(
    len(idx_valid)
)

test_count = int(
    len(idx_test)
)


assert (
    train_count
    + valid_count
    + test_count
    == num_nodes
)


train_pct = (
    train_count
    / num_nodes
    * 100.0
)

valid_pct = (
    valid_count
    / num_nodes
    * 100.0
)

test_pct = (
    test_count
    / num_nodes
    * 100.0
)


print(
    "\n===== SPLIT SIZES ====="
)

print(
    "Train:",
    f"{train_count:,}",
    f"({train_pct:.6f}%)"
)

print(
    "Validation:",
    f"{valid_count:,}",
    f"({valid_pct:.6f}%)"
)

print(
    "Test:",
    f"{test_count:,}",
    f"({test_pct:.6f}%)"
)


# ============================================================
# 7. CLASS COUNT HELPER
# ============================================================

def class_counts(y):

    values, counts = np.unique(
        y,
        return_counts=True
    )

    result = {
        int(v): int(c)
        for v, c
        in zip(
            values,
            counts
        )
    }

    return result


train_classes = class_counts(
    y_train
)

valid_classes = class_counts(
    y_valid
)

test_classes = class_counts(
    y_test
)


# ============================================================
# 8. CLASS DISTRIBUTION BY SPLIT
# ============================================================

print(
    "\n===== CLASS DISTRIBUTION ====="
)


print(
    "TRAIN"
)

print(
    " Normal:",
    f"{train_classes[0]:,}"
)

print(
    " Anomaly:",
    f"{train_classes[1]:,}"
)

print(
    " Anomaly %:",
    f"{train_classes[1] / train_count * 100:.6f}%"
)


print(
    "\nVALIDATION"
)

print(
    " Normal:",
    f"{valid_classes[0]:,}"
)

print(
    " Anomaly:",
    f"{valid_classes[1]:,}"
)

print(
    " Anomaly %:",
    f"{valid_classes[1] / valid_count * 100:.6f}%"
)


print(
    "\nTEST"
)

print(
    " Normal:",
    f"{test_classes[0]:,}"
)

print(
    " Anomaly:",
    f"{test_classes[1]:,}"
)

print(
    " Anomaly %:",
    f"{test_classes[1] / test_count * 100:.6f}%"
)


# ============================================================
# 9. VERIFY EXPECTED EXACT COUNTS
# ============================================================

# These counts follow directly from sklearn using the
# official BWGNN split settings on the verified T-Social labels.

expected_train = 2_312_426

expected_valid = 1_144_650

expected_test = 2_323_989


assert (
    train_count == expected_train
)

assert (
    valid_count == expected_valid
)

assert (
    test_count == expected_test
)


expected_train_normal = 2_242_714
expected_train_anomaly = 69_712

expected_valid_normal = 1_110_143
expected_valid_anomaly = 34_507

expected_test_normal = 2_253_928
expected_test_anomaly = 70_061


assert (
    train_classes[0]
    == expected_train_normal
)

assert (
    train_classes[1]
    == expected_train_anomaly
)

assert (
    valid_classes[0]
    == expected_valid_normal
)

assert (
    valid_classes[1]
    == expected_valid_anomaly
)

assert (
    test_classes[0]
    == expected_test_normal
)

assert (
    test_classes[1]
    == expected_test_anomaly
)


print(
    "\nPASS: Exact reference split counts verified."
)


# ============================================================
# 10. CHECK NO INDEX OVERLAP
# ============================================================

# Sorting is not needed.
# Boolean membership arrays give a memory-efficient
# exact overlap/completeness check.

membership = np.zeros(
    num_nodes,
    dtype=np.uint8
)


membership[
    idx_train
] += 1

membership[
    idx_valid
] += 1

membership[
    idx_test
] += 1


assert np.all(
    membership == 1
)


print(
    "PASS: Every node belongs to exactly one split."
)

del membership


# ============================================================
# 11. SAVE EXACT SPLIT INDICES
# ============================================================

TRAIN_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_reference_train_indices.npy"
)

VALID_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_reference_valid_indices.npy"
)

TEST_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_reference_test_indices.npy"
)


np.save(
    TRAIN_PATH,
    idx_train.astype(
        np.int32,
        copy=False
    )
)

np.save(
    VALID_PATH,
    idx_valid.astype(
        np.int32,
        copy=False
    )
)

np.save(
    TEST_PATH,
    idx_test.astype(
        np.int32,
        copy=False
    )
)


print(
    "\nPASS: Exact reference split indices saved."
)


# ============================================================
# 12. RECORD SPLIT METADATA
# ============================================================

task7_results = {

    "dataset":
        "T-Social",

    "split_role":
        "Original/reference BWGNN reproduction split",

    "split_type":
        "Random stratified node split",

    "temporal":
        False,

    "shuffle":
        True,

    "random_state":
        2,

    "first_split_train_size":
        0.4,

    "second_split_test_size":
        0.67,

    "train_nodes":
        train_count,

    "validation_nodes":
        valid_count,

    "test_nodes":
        test_count,

    "train_percent":
        float(train_pct),

    "validation_percent":
        float(valid_pct),

    "test_percent":
        float(test_pct),

    "train_normal":
        int(train_classes[0]),

    "train_anomaly":
        int(train_classes[1]),

    "validation_normal":
        int(valid_classes[0]),

    "validation_anomaly":
        int(valid_classes[1]),

    "test_normal":
        int(test_classes[0]),

    "test_anomaly":
        int(test_classes[1]),

    "split_source":
        "Official BWGNN main.py",

    "fixed_in_dataset_file":
        False,

    "generated_by_code":
        True,

    "important_note":
        (
            "The reference split is approximately "
            "40.0% train / 19.8% validation / "
            "40.2% test, not exactly 40/20/40."
        ),

    "benchmark_note":
        (
            "Keep this reproduction/reference split "
            "separate from any later unified split "
            "used for cross-dataset fair comparison."
        )
}


TASK7_JSON = os.path.join(
    OUTPUT_DIR,
    "tsocial_task7_reference_split.json"
)


with open(
    TASK7_JSON,
    "w"
) as f:

    json.dump(
        task7_results,
        f,
        indent=4
    )


print(
    "PASS: Task 7 metadata saved."
)


# ============================================================
# 13. FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 7 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nValues to record:"
)

print(
    "Split type = Random stratified"
)

print(
    "Temporal split = No"
)

print(
    "Random state = 2"
)

print(
    "Train =",
    f"{train_count:,}",
    f"({train_pct:.6f}%)"
)

print(
    "Validation =",
    f"{valid_count:,}",
    f"({valid_pct:.6f}%)"
)

print(
    "Test =",
    f"{test_count:,}",
    f"({test_pct:.6f}%)"
)


print(
    "\nTrain normal/anomaly =",
    f"{train_classes[0]:,}",
    "/",
    f"{train_classes[1]:,}"
)

print(
    "Validation normal/anomaly =",
    f"{valid_classes[0]:,}",
    "/",
    f"{valid_classes[1]:,}"
)

print(
    "Test normal/anomaly =",
    f"{test_classes[0]:,}",
    "/",
    f"{test_classes[1]:,}"
)


print(
    "\nIMPORTANT:"
)

print(
    "This is the BWGNN reference/reproduction split."
)

print(
    "Do not confuse it with a future unified "
    "cross-dataset benchmark split."
)

===== T-SOCIAL TASK 7 =====

Nodes: 5,781,065
Labels shape: (5781065,)
Labels dtype: int8

PASS: Actual T-Social labels verified.

PASS: First 40% stratified split created.
PASS: Validation/test split created.

===== SPLIT SIZES =====
Train: 2,312,426 (40.000000%)
Validation: 1,144,650 (19.799985%)
Test: 2,323,989 (40.200015%)

===== CLASS DISTRIBUTION =====
TRAIN
 Normal: 2,242,714
 Anomaly: 69,712
 Anomaly %: 3.014669%

VALIDATION
 Normal: 1,110,143
 Anomaly: 34,507
 Anomaly %: 3.014633%

TEST
 Normal: 2,253,928
 Anomaly: 70,061
 Anomaly %: 3.014687%

PASS: Exact reference split counts verified.
PASS: Every node belongs to exactly one split.

PASS: Exact reference split indices saved.
PASS: Task 7 metadata saved.

T-SOCIAL TASK 7 COMPLETE

Values to record:
Split type = Random stratified
Temporal split = No
Random state = 2
Train = 2,312,426 (40.000000%)
Validation = 1,144,650 (19.799985%)
Test = 2,323,989 (40.200015%)

Train normal/anomaly = 2,242,714 / 69,712
Validation normal/anom

## Task 8 — Record sources, provenance, compatibility and limitations

This step documents where T-Social comes from, which source was actually used,
and the main limitations that should be considered when comparing T-Social
with other graph-fraud datasets.

### Primary sources

- Paper: Tang et al. (2022), *Rethinking Graph Neural Networks for Anomaly
  Detection*, ICML 2022.
- Official implementation: BWGNN GitHub repository.
- Official dataset release: Google Drive folder linked by the BWGNN authors.
- Secondary benchmark reference: GADBench.

### Dataset source used in this analysis

The analysis uses the T-Social file extracted directly from the authors'
official `tsocial.zip` release.

The official extracted graph was independently verified during this work:

- archive size: 744,239,223 bytes
- extracted graph size: 4,110,300,257 bytes
- SHA256:
  `8d577114cff12f7de35eda2974f17825ef9febe20c661e95e8d21072a6dfc8d0`

A separate Kaggle mirror was investigated but rejected as the authoritative
source because:

- its size differed from the official graph,
- its SHA256 differed from the official graph,
- it could not be deserialized successfully,
- while the official authors' graph loaded successfully.

Therefore, all final T-Social statistics in this project come from the
authors' official release.

### Compatibility

T-Social is represented as:

- a homogeneous graph,
- one semantic node type,
- one relation type,
- a static graph benchmark,
- binary node anomaly classification.

It is directly suitable for homogeneous and single-relation graph anomaly
detection methods.

Models designed specifically for heterogeneous or multi-relational graphs
receive no relation-type diversity from T-Social unless the model or input
representation is adapted.

The original BWGNN implementation specifies PyTorch 1.9.0 and DGL 0.8.1.
These versions were therefore used as the reproducibility environment for
reading the original graph.

### Important limitations

1. Account identities are anonymized.
2. The positive class represents general anomalous accounts rather than only
   one fraud subtype.
3. Example anomaly categories include fraud, money laundering and online
   gambling.
4. The paper describes the 10-dimensional account features at a broad level,
   including registration, login/activity and interaction information.
5. The released benchmark is static; the three-month friendship rule is an
   edge-selection condition rather than a temporal graph sequence.
6. The graph is very large, so full-graph processing has substantial memory
   and computation requirements.
7. The official DGL graph stores every friendship in both directions:
   146,211,016 stored directed entries correspond to 73,105,508 verified
   unique undirected friendships.
8. The original dataset file contains labels and features but no fixed
   train/validation/test masks. The BWGNN reference split is generated by code.
9. Dataset mirrors should not automatically be assumed to be identical to the
   authors' release; source integrity should be verified.

In [10]:
# CELL 8 — T-SOCIAL TASK 8
# SOURCES, PROVENANCE, COMPATIBILITY AND LIMITATIONS

import os
import json


print("===== T-SOCIAL TASK 8 =====")


# ============================================================
# 1. PRIMARY SOURCE INFORMATION
# ============================================================

paper = (
    "Tang et al. (2022), "
    "Rethinking Graph Neural Networks for Anomaly Detection, "
    "ICML 2022"
)

official_repository = (
    "squareRoot3/Rethinking-Anomaly-Detection"
)

official_download_source = (
    "Authors' Google Drive dataset folder linked "
    "from the official BWGNN repository"
)

secondary_benchmark = (
    "squareRoot3/GADBench"
)


print("\n===== PRIMARY SOURCES =====")

print(
    "Paper:",
    paper
)

print(
    "Official implementation:",
    official_repository
)

print(
    "Official dataset source:",
    official_download_source
)

print(
    "Secondary benchmark:",
    secondary_benchmark
)


# ============================================================
# 2. VERIFIED SOURCE PROVENANCE
# ============================================================

official_archive_name = (
    "tsocial.zip"
)

official_archive_size = (
    744_239_223
)

official_graph_size = (
    4_110_300_257
)

official_graph_sha256 = (
    "8d577114cff12f7de35eda2974f17825"
    "ef9febe20c661e95e8d21072a6dfc8d0"
)


source_status = (
    "Authors' official release used"
)


print(
    "\n===== VERIFIED DATASET PROVENANCE ====="
)

print(
    "Source status:",
    source_status
)

print(
    "Official archive:",
    official_archive_name
)

print(
    "Official archive size:",
    f"{official_archive_size:,}",
    "bytes"
)

print(
    "Official extracted graph size:",
    f"{official_graph_size:,}",
    "bytes"
)

print(
    "Official graph SHA256:"
)

print(
    official_graph_sha256
)


# ============================================================
# 3. KAGGLE MIRROR AUDIT
# ============================================================

kaggle_mirror_checked = True

kaggle_mirror_size = (
    2_218_311_680
)

kaggle_mirror_sha256 = (
    "1805a980a8cbb14e57b6dadb1916887"
    "c969707cb27bd25be1d0d5f9abf438c85"
)

kaggle_same_size_as_official = (
    False
)

kaggle_same_hash_as_official = (
    False
)

kaggle_mirror_used = (
    False
)


print(
    "\n===== KAGGLE MIRROR AUDIT ====="
)

print(
    "Mirror checked:",
    kaggle_mirror_checked
)

print(
    "Mirror size:",
    f"{kaggle_mirror_size:,}",
    "bytes"
)

print(
    "Mirror SHA256:"
)

print(
    kaggle_mirror_sha256
)

print(
    "Same size as official:",
    kaggle_same_size_as_official
)

print(
    "Same SHA256 as official:",
    kaggle_same_hash_as_official
)

print(
    "Used for final analysis:",
    kaggle_mirror_used
)


assert (
    kaggle_mirror_size
    != official_graph_size
)

assert (
    kaggle_mirror_sha256
    != official_graph_sha256
)


print(
    "\nPASS: Kaggle mirror is correctly excluded "
    "from the authoritative analysis."
)


# ============================================================
# 4. ORIGINAL SOFTWARE ENVIRONMENT
# ============================================================

original_environment = {

    "python_used_for_reproduction":
        "3.9.23",

    "pytorch":
        "1.9.0+cpu",

    "dgl":
        "0.8.1",

    "numpy":
        "1.23.5",

    "scipy":
        "1.10.1"
}


print(
    "\n===== REPRODUCIBILITY ENVIRONMENT ====="
)


for key, value in original_environment.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# 5. STRUCTURAL COMPATIBILITY
# ============================================================

graph_type = (
    "Homogeneous"
)

relation_structure = (
    "Single-relational"
)

semantic_node_type = (
    "Anonymized social-network account"
)

semantic_relation = (
    "Social friendship"
)

temporal_type = (
    "Static"
)

prediction_task = (
    "Binary node anomaly classification"
)


print(
    "\n===== MODEL COMPATIBILITY ====="
)

print(
    "Graph type:",
    graph_type
)

print(
    "Relation structure:",
    relation_structure
)

print(
    "Node type:",
    semantic_node_type
)

print(
    "Relation:",
    semantic_relation
)

print(
    "Temporal structure:",
    temporal_type
)

print(
    "Prediction task:",
    prediction_task
)


compatible_with = [
    "Homogeneous GNNs",
    "Single-relation graph models",
    "Node-level graph anomaly detection models",
    "Binary node classification models"
]


requires_adaptation_or_has_no_extra_relation_information = [
    "Heterogeneous GNN architectures",
    "Multi-relational GNN architectures",
    "Temporal/dynamic graph models requiring timestamps"
]


print(
    "\nNaturally compatible with:"
)

for item in compatible_with:

    print(
        " -",
        item
    )


print(
    "\nSpecial consideration for:"
)

for item in (
    requires_adaptation_or_has_no_extra_relation_information
):

    print(
        " -",
        item
    )


# ============================================================
# 6. VERIFIED GRAPH CHARACTERISTICS
# ============================================================

verified_characteristics = {

    "nodes":
        5_781_065,

    "unique_undirected_edges":
        73_105_508,

    "raw_stored_dgl_entries":
        146_211_016,

    "features":
        10,

    "normal_nodes":
        5_606_785,

    "anomaly_nodes":
        174_280,

    "anomaly_percentage":
        3.014669,

    "global_heterophily":
        0.376095095,

    "local_h_mean":
        0.122186690,

    "normal_mean_local_h":
        0.100309969,

    "anomaly_mean_local_h":
        0.825985563,

    "isolates":
        0
}


# ============================================================
# 7. DATASET LIMITATIONS
# ============================================================

limitations = [

    (
        "Account identities are anonymized, so the graph "
        "cannot be mapped back to identifiable users."
    ),

    (
        "The positive class represents anomalous accounts "
        "rather than one single fraud subtype."
    ),

    (
        "Reported anomalous categories include fraud, "
        "money laundering and online gambling."
    ),

    (
        "The paper describes the 10-dimensional features "
        "at a broad semantic level rather than exposing "
        "real account identities."
    ),

    (
        "The released benchmark is static; the three-month "
        "friendship condition is an edge-selection rule "
        "rather than a temporal sequence."
    ),

    (
        "The graph is very large, creating substantial "
        "memory and computation requirements for "
        "full-graph analysis."
    ),

    (
        "The official DGL representation stores each "
        "verified undirected friendship in both directions."
    ),

    (
        "The source graph does not contain fixed "
        "train/validation/test masks; the reference "
        "BWGNN split is generated by code."
    ),

    (
        "Unofficial mirrors should not automatically be "
        "treated as equivalent to the authors' release; "
        "source integrity should be checked."
    )
]


print(
    "\n===== LIMITATIONS ====="
)


for number, item in enumerate(
    limitations,
    start=1
):

    print(
        f"{number}. {item}"
    )


# ============================================================
# 8. IMPORTANT EDGE-STORAGE NOTE
# ============================================================

raw_edges = (
    146_211_016
)

undirected_edges = (
    73_105_508
)


assert (
    raw_edges
    == 2 * undirected_edges
)


edge_storage_note = (
    "The official DGL graph stores 146,211,016 directed "
    "entries. Structural verification found no self-loops, "
    "no parallel directed edges and complete reciprocity, "
    "so these correspond to 73,105,508 unique undirected "
    "friendships."
)


print(
    "\n===== EDGE STORAGE NOTE ====="
)

print(
    edge_storage_note
)


# ============================================================
# 9. SPLIT PROVENANCE NOTE
# ============================================================

split_note = (
    "The official graph file has no fixed split masks. "
    "The BWGNN reference split is generated using a "
    "random stratified split with random_state=2: "
    "40.0% train, approximately 19.8% validation and "
    "40.2% test."
)


print(
    "\n===== SPLIT NOTE ====="
)

print(
    split_note
)


# ============================================================
# 10. SAVE TASK 8
# ============================================================

task8_results = {

    "dataset":
        "T-Social",

    "paper":
        paper,

    "official_repository":
        official_repository,

    "official_download_source":
        official_download_source,

    "secondary_benchmark":
        secondary_benchmark,

    "source_status":
        source_status,

    "official_archive_name":
        official_archive_name,

    "official_archive_size_bytes":
        official_archive_size,

    "official_graph_size_bytes":
        official_graph_size,

    "official_graph_sha256":
        official_graph_sha256,

    "kaggle_mirror_checked":
        kaggle_mirror_checked,

    "kaggle_mirror_size_bytes":
        kaggle_mirror_size,

    "kaggle_mirror_sha256":
        kaggle_mirror_sha256,

    "kaggle_mirror_same_as_official":
        False,

    "kaggle_mirror_used":
        False,

    "reproducibility_environment":
        original_environment,

    "graph_type":
        graph_type,

    "relation_structure":
        relation_structure,

    "semantic_node_type":
        semantic_node_type,

    "semantic_relation":
        semantic_relation,

    "temporal_type":
        temporal_type,

    "prediction_task":
        prediction_task,

    "compatible_with":
        compatible_with,

    "special_consideration_for":
        requires_adaptation_or_has_no_extra_relation_information,

    "verified_characteristics":
        verified_characteristics,

    "limitations":
        limitations,

    "edge_storage_note":
        edge_storage_note,

    "split_note":
        split_note
}


OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


TASK8_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task8_sources_limitations.json"
)


with open(
    TASK8_PATH,
    "w"
) as f:

    json.dump(
        task8_results,
        f,
        indent=4
    )


print(
    "\nPASS: Task 8 results saved:"
)

print(
    TASK8_PATH
)


# ============================================================
# 11. FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 8 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nValues to record:"
)

print(
    "Primary source = Authors' official BWGNN release"
)

print(
    "Official implementation = "
    "squareRoot3/Rethinking-Anomaly-Detection"
)

print(
    "Paper = Tang et al. (2022), ICML"
)

print(
    "Graph compatibility = "
    "Homogeneous / single-relational / static"
)

print(
    "Original DGL environment = "
    "PyTorch 1.9.0 + DGL 0.8.1"
)

print(
    "Authoritative mirror status = "
    "Kaggle copy rejected; official release used"
)

print(
    "Comparable edges = 73,105,508 "
    "unique undirected friendships"
)

===== T-SOCIAL TASK 8 =====

===== PRIMARY SOURCES =====
Paper: Tang et al. (2022), Rethinking Graph Neural Networks for Anomaly Detection, ICML 2022
Official implementation: squareRoot3/Rethinking-Anomaly-Detection
Official dataset source: Authors' Google Drive dataset folder linked from the official BWGNN repository
Secondary benchmark: squareRoot3/GADBench

===== VERIFIED DATASET PROVENANCE =====
Source status: Authors' official release used
Official archive: tsocial.zip
Official archive size: 744,239,223 bytes
Official extracted graph size: 4,110,300,257 bytes
Official graph SHA256:
8d577114cff12f7de35eda2974f17825ef9febe20c661e95e8d21072a6dfc8d0

===== KAGGLE MIRROR AUDIT =====
Mirror checked: True
Mirror size: 2,218,311,680 bytes
Mirror SHA256:
1805a980a8cbb14e57b6dadb1916887c969707cb27bd25be1d0d5f9abf438c85
Same size as official: False
Same SHA256 as official: False
Used for final analysis: False

PASS: Kaggle mirror is correctly excluded from the authoritative analysis.

===== 

## Task 9 — Create the final common-table row for T-Social

This step combines the verified results from Tasks 1–8 into the common dataset
comparison format used for the project.

For cross-dataset consistency, the edge count reported in the common table is
the verified **unique undirected non-self edge count**, not the raw number of
stored DGL edge entries.

Therefore:

- Raw DGL stored entries: 146,211,016
- Comparable unique undirected edges: 73,105,508

The positive class is reported as **fraud/anomaly** because T-Social contains
general anomalous accounts, including categories such as fraud, money
laundering and online gambling.

The final row records:

- dataset domain,
- nodes,
- comparable edges,
- feature dimension,
- anomaly and normal counts,
- anomaly percentage,
- class imbalance,
- graph structure,
- node and relation types,
- static/dynamic status,
- real-world versus injected anomaly origin.

The resulting row is saved as both CSV and JSON so it can later be merged with
the corresponding rows for YelpChi and the remaining benchmark datasets.

In [11]:
# CELL 9 — T-SOCIAL TASK 9
# FINAL COMMON-TABLE ROW
#
# Uses the same common-table structure as YelpChi.

import os
import json
import pandas as pd


print("===== T-SOCIAL TASK 9 =====")


# ============================================================
# 1. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 2. LOAD VERIFIED TASK 5 RESULT
# ============================================================

TASK5_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task5_global_heterophily.json"
)


if not os.path.isfile(TASK5_PATH):

    raise FileNotFoundError(
        "Task 5 result was not found:\n"
        f"{TASK5_PATH}"
    )


with open(
    TASK5_PATH,
    "r"
) as f:

    task5 = json.load(f)


print(
    "\nPASS: Task 5 structural results loaded."
)


# ============================================================
# 3. LOAD VERIFIED TASK 6 RESULT
# ============================================================

TASK6_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task6_local_heterophily.json"
)


if not os.path.isfile(TASK6_PATH):

    raise FileNotFoundError(
        "Task 6 result was not found:\n"
        f"{TASK6_PATH}"
    )


with open(
    TASK6_PATH,
    "r"
) as f:

    task6 = json.load(f)


print(
    "PASS: Task 6 local-heterophily results loaded."
)


# ============================================================
# 4. LOAD VERIFIED TASK 7 RESULT
# ============================================================

TASK7_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task7_reference_split.json"
)


if not os.path.isfile(TASK7_PATH):

    raise FileNotFoundError(
        "Task 7 result was not found:\n"
        f"{TASK7_PATH}"
    )


with open(
    TASK7_PATH,
    "r"
) as f:

    task7 = json.load(f)


print(
    "PASS: Task 7 reference-split results loaded."
)


# ============================================================
# 5. LOAD VERIFIED TASK 8 RESULT
# ============================================================

TASK8_PATH = os.path.join(
    OUTPUT_DIR,
    "tsocial_task8_sources_limitations.json"
)


if not os.path.isfile(TASK8_PATH):

    raise FileNotFoundError(
        "Task 8 result was not found:\n"
        f"{TASK8_PATH}"
    )


with open(
    TASK8_PATH,
    "r"
) as f:

    task8 = json.load(f)


print(
    "PASS: Task 8 provenance results loaded."
)


# ============================================================
# 6. VERIFIED CORE NUMERICAL VALUES
# ============================================================

nodes = (
    5_781_065
)

raw_stored_edges = int(
    task5[
        "raw_stored_dgl_edge_entries"
    ]
)

comparable_edges = int(
    task5[
        "unique_undirected_nonself_edges"
    ]
)

features = (
    10
)

normal_nodes = (
    5_606_785
)

anomaly_nodes = (
    174_280
)


anomaly_percentage = (
    anomaly_nodes
    / nodes
    * 100.0
)


imbalance_ratio = (
    normal_nodes
    / anomaly_nodes
)


# ============================================================
# 7. VERIFY CORE NUMBERS
# ============================================================

assert (
    nodes
    == 5_781_065
)

assert (
    raw_stored_edges
    == 146_211_016
)

assert (
    comparable_edges
    == 73_105_508
)

assert (
    raw_stored_edges
    == 2 * comparable_edges
)

assert (
    normal_nodes
    + anomaly_nodes
    == nodes
)

assert (
    int(
        task5[
            "self_loop_entries"
        ]
    )
    == 0
)

assert (
    int(
        task5[
            "missing_reverse_entries"
        ]
    )
    == 0
)

assert (
    task5[
        "is_multigraph"
    ]
    is False
)


print(
    "\nPASS: Core numerical values verified."
)

print(
    "PASS: Comparable edge count verified."
)


# ============================================================
# 8. VERIFY HETEROPHILY RESULTS
# ============================================================

global_h = float(
    task5[
        "global_heterophily"
    ]
)

local_h_mean = float(
    task6[
        "mean"
    ]
)

normal_local_h = float(
    task6[
        "normal_mean_local_h"
    ]
)

anomaly_local_h = float(
    task6[
        "anomaly_mean_local_h"
    ]
)

isolates = int(
    task6[
        "isolates"
    ]
)


assert abs(
    global_h
    - 0.376095095324
) < 1e-12


assert (
    isolates == 0
)


print(
    "PASS: Heterophily results verified."
)


# ============================================================
# 9. STRUCTURAL / SEMANTIC VALUES
# ============================================================

dataset_name = (
    "T-Social"
)

domain = (
    "Social-network anomalous-account detection"
)

graph_type = (
    "Homogeneous; single-relational"
)

node_types = (
    "1 (anonymized social-network account)"
)

relation_types = (
    "1 (social friendship)"
)

static_dynamic = (
    "Static"
)

organic_injected = (
    "Real-world / human-expert annotated; "
    "not synthetic injection"
)


# ============================================================
# 10. FORMAT FINAL COMMON-TABLE VALUES
# ============================================================

edges_for_table = (
    f"{comparable_edges:,} "
    "unique undirected edges"
)

fraud_anomaly_percentage_for_table = (
    f"{anomaly_percentage:.6f}%"
)

imbalance_for_table = (
    f"{imbalance_ratio:.6f}:1 "
    "normal:anomaly"
)


# ============================================================
# 11. CREATE FINAL COMMON-TABLE ROW
#
# IMPORTANT:
# Keep these headings in this exact order so the row can
# later be concatenated directly with YelpChi.
# ============================================================

common_table_row = {

    "Dataset":
        dataset_name,

    "Domain":
        domain,

    "Nodes":
        nodes,

    "Edges":
        edges_for_table,

    "Features":
        features,

    "Fraud/anomaly nodes":
        anomaly_nodes,

    "Normal nodes":
        normal_nodes,

    "Fraud/anomaly %":
        fraud_anomaly_percentage_for_table,

    "Imbalance ratio":
        imbalance_for_table,

    "Graph type":
        graph_type,

    "Node types":
        node_types,

    "Relation types":
        relation_types,

    "Static/dynamic":
        static_dynamic,

    "Organic/injected":
        organic_injected
}


common_table_df = pd.DataFrame(
    [
        common_table_row
    ]
)


# ============================================================
# 12. DISPLAY FINAL ROW
# ============================================================

print(
    "\n===== FINAL T-SOCIAL COMMON-TABLE ROW ====="
)

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_colwidth",
    120
)


print(
    common_table_df.to_string(
        index=False
    )
)


# ============================================================
# 13. SAVE COMMON-TABLE CSV
# ============================================================

COMMON_CSV = os.path.join(
    OUTPUT_DIR,
    "tsocial_common_table_row.csv"
)


common_table_df.to_csv(
    COMMON_CSV,
    index=False
)


print(
    "\nPASS: Common-table CSV saved:"
)

print(
    COMMON_CSV
)


# ============================================================
# 14. SAVE COMMON-TABLE JSON
# ============================================================

COMMON_JSON = os.path.join(
    OUTPUT_DIR,
    "tsocial_common_table_row.json"
)


with open(
    COMMON_JSON,
    "w"
) as f:

    json.dump(
        common_table_row,
        f,
        indent=4
    )


print(
    "PASS: Common-table JSON saved:"
)

print(
    COMMON_JSON
)


# ============================================================
# 15. CREATE DETAILED SUPPORTING SUMMARY
#
# These fields are not part of the compact common table,
# but preserve the important calculations supporting it.
# ============================================================

detailed_summary = {

    "dataset":
        "T-Social",

    # --------------------------------------------------------
    # Numerical characteristics
    # --------------------------------------------------------

    "nodes":
        nodes,

    "raw_stored_dgl_edge_entries":
        raw_stored_edges,

    "comparable_unique_undirected_edges":
        comparable_edges,

    "features":
        features,

    "normal_nodes":
        normal_nodes,

    "anomaly_nodes":
        anomaly_nodes,

    "anomaly_percentage":
        float(
            anomaly_percentage
        ),

    "normal_to_anomaly_ratio":
        float(
            imbalance_ratio
        ),

    # --------------------------------------------------------
    # Structure
    # --------------------------------------------------------

    "graph_type":
        "Homogeneous",

    "relation_structure":
        "Single-relational",

    "node_type":
        "Anonymized social-network account",

    "relation_type":
        "Social friendship",

    "temporal_type":
        "Static",

    "self_loops":
        int(
            task5[
                "self_loop_entries"
            ]
        ),

    "parallel_directed_edges":
        bool(
            task5[
                "is_multigraph"
            ]
        ),

    "missing_reverse_edges":
        int(
            task5[
                "missing_reverse_entries"
            ]
        ),

    "fully_reciprocal":
        bool(
            task5[
                "fully_reciprocal"
            ]
        ),

    # --------------------------------------------------------
    # Anomaly provenance
    # --------------------------------------------------------

    "anomaly_origin":
        (
            "Real-world / non-injected"
        ),

    "label_source":
        (
            "Human-expert annotation"
        ),

    "positive_class":
        (
            "Anomalous account"
        ),

    "example_categories": [
        "Fraud",
        "Money laundering",
        "Online gambling"
    ],

    # --------------------------------------------------------
    # Global heterophily
    # --------------------------------------------------------

    "homophilic_undirected_edges":
        int(
            task5[
                "homophilic_undirected_edges"
            ]
        ),

    "heterophilic_undirected_edges":
        int(
            task5[
                "heterophilic_undirected_edges"
            ]
        ),

    "global_heterophily":
        global_h,

    # --------------------------------------------------------
    # Local heterophily
    # --------------------------------------------------------

    "local_h_mean":
        local_h_mean,

    "local_h_median":
        float(
            task6[
                "median"
            ]
        ),

    "local_h_population_std":
        float(
            task6[
                "population_std"
            ]
        ),

    "local_h_q1":
        float(
            task6[
                "q1"
            ]
        ),

    "local_h_q3":
        float(
            task6[
                "q3"
            ]
        ),

    "local_h_minimum":
        float(
            task6[
                "minimum"
            ]
        ),

    "local_h_maximum":
        float(
            task6[
                "maximum"
            ]
        ),

    "normal_mean_local_h":
        normal_local_h,

    "anomaly_mean_local_h":
        anomaly_local_h,

    "isolates":
        isolates,

    # --------------------------------------------------------
    # Reference split
    # --------------------------------------------------------

    "reference_split_type":
        task7[
            "split_type"
        ],

    "reference_split_random_state":
        int(
            task7[
                "random_state"
            ]
        ),

    "train_nodes":
        int(
            task7[
                "train_nodes"
            ]
        ),

    "validation_nodes":
        int(
            task7[
                "validation_nodes"
            ]
        ),

    "test_nodes":
        int(
            task7[
                "test_nodes"
            ]
        ),

    "train_percent":
        float(
            task7[
                "train_percent"
            ]
        ),

    "validation_percent":
        float(
            task7[
                "validation_percent"
            ]
        ),

    "test_percent":
        float(
            task7[
                "test_percent"
            ]
        ),

    # --------------------------------------------------------
    # Provenance
    # --------------------------------------------------------

    "source_status":
        task8[
            "source_status"
        ],

    "official_graph_sha256":
        task8[
            "official_graph_sha256"
        ],

    "official_repository":
        task8[
            "official_repository"
        ],

    "kaggle_mirror_used":
        task8[
            "kaggle_mirror_used"
        ]
}


DETAIL_JSON = os.path.join(
    OUTPUT_DIR,
    "tsocial_task9_detailed_summary.json"
)


with open(
    DETAIL_JSON,
    "w"
) as f:

    json.dump(
        detailed_summary,
        f,
        indent=4
    )


print(
    "PASS: Detailed supporting summary saved:"
)

print(
    DETAIL_JSON
)


# ============================================================
# 16. FINAL VALIDATION
# ============================================================

expected_columns = [

    "Dataset",
    "Domain",
    "Nodes",
    "Edges",
    "Features",
    "Fraud/anomaly nodes",
    "Normal nodes",
    "Fraud/anomaly %",
    "Imbalance ratio",
    "Graph type",
    "Node types",
    "Relation types",
    "Static/dynamic",
    "Organic/injected"
]


assert (
    list(
        common_table_df.columns
    )
    == expected_columns
)


assert (
    len(common_table_df)
    == 1
)


print(
    "\nPASS: Common-table column order verified."
)

print(
    "PASS: T-Social row is ready to merge "
    "directly with the YelpChi row."
)


# ============================================================
# 17. FINAL REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 9 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nFinal common-table values:"
)

print(
    "Dataset = T-Social"
)

print(
    "Domain = Social-network "
    "anomalous-account detection"
)

print(
    "Nodes =",
    f"{nodes:,}"
)

print(
    "Edges =",
    f"{comparable_edges:,}",
    "unique undirected edges"
)

print(
    "Features =",
    features
)

print(
    "Fraud/anomaly nodes =",
    f"{anomaly_nodes:,}"
)

print(
    "Normal nodes =",
    f"{normal_nodes:,}"
)

print(
    "Fraud/anomaly % =",
    f"{anomaly_percentage:.6f}%"
)

print(
    "Imbalance ratio =",
    f"{imbalance_ratio:.6f}:1 "
    "normal:anomaly"
)

print(
    "Graph type = Homogeneous; "
    "single-relational"
)

print(
    "Node types = 1 "
    "(anonymized social-network account)"
)

print(
    "Relation types = 1 "
    "(social friendship)"
)

print(
    "Static/dynamic = Static"
)

print(
    "Organic/injected = "
    "Real-world / human-expert annotated; "
    "not synthetic injection"
)

===== T-SOCIAL TASK 9 =====

PASS: Task 5 structural results loaded.
PASS: Task 6 local-heterophily results loaded.
PASS: Task 7 reference-split results loaded.
PASS: Task 8 provenance results loaded.

PASS: Core numerical values verified.
PASS: Comparable edge count verified.
PASS: Heterophily results verified.

===== FINAL T-SOCIAL COMMON-TABLE ROW =====
 Dataset                                     Domain   Nodes                              Edges  Features  Fraud/anomaly nodes  Normal nodes Fraud/anomaly %            Imbalance ratio                     Graph type                            Node types        Relation types Static/dynamic                                             Organic/injected
T-Social Social-network anomalous-account detection 5781065 73,105,508 unique undirected edges        10               174280       5606785       3.014669% 32.171133:1 normal:anomaly Homogeneous; single-relational 1 (anonymized social-network account) 1 (social friendship)         Static Re

## Task 10 — Final validation and export package

This final step audits the completed T-Social analysis and creates a
reproducibility package.

No graph calculations are repeated in this step.

The saved outputs from the earlier tasks are checked against one another for:

- node and class-count consistency,
- comparable edge count,
- reciprocal-edge verification,
- global heterophily,
- local heterophily,
- isolate count,
- reference split counts,
- source provenance,
- final common-table values.

A SHA256 manifest is created for the exported analysis files.

The final ZIP package contains the analytical results and reproducibility
artifacts, but does not include:

- the 4.11 GB original T-Social graph,
- the 744 MB source ZIP,
- the temporary legacy Python environment.

The official source-file SHA256 is preserved in the metadata so the source can
still be independently verified.

The final package therefore provides a compact record of the T-Social dataset
analysis without unnecessarily duplicating the original multi-gigabyte data.

In [12]:
# CELL 10 — T-SOCIAL TASK 10
# FINAL AUDIT + REPRODUCIBILITY EXPORT PACKAGE

import os
import json
import csv
import hashlib
import zipfile
import shutil
import pandas as pd
import numpy as np


print("===== T-SOCIAL TASK 10 =====")


# ============================================================
# 1. ANALYSIS DIRECTORY
# ============================================================

ANALYSIS_DIR = (
    "/kaggle/working/"
    "tsocial_analysis"
)

if not os.path.isdir(ANALYSIS_DIR):

    raise FileNotFoundError(
        "T-Social analysis directory is missing:\n"
        f"{ANALYSIS_DIR}"
    )


print("\nAnalysis directory:")
print(ANALYSIS_DIR)


# ============================================================
# 2. REQUIRED CORE FILES
# ============================================================

required_core_files = {

    "Task 3 structure":
        "tsocial_task3_structure.json",

    "Task 4 anomaly provenance":
        "tsocial_task4_anomaly_origin.json",

    "Task 5 global heterophily":
        "tsocial_task5_global_heterophily.json",

    "Task 6 local heterophily":
        "tsocial_task6_local_heterophily.json",

    "Task 7 reference split":
        "tsocial_task7_reference_split.json",

    "Task 8 sources / limitations":
        "tsocial_task8_sources_limitations.json",

    "Task 9 common row CSV":
        "tsocial_common_table_row.csv",

    "Task 9 common row JSON":
        "tsocial_common_table_row.json",

    "Task 9 detailed summary":
        "tsocial_task9_detailed_summary.json",

    "Labels":
        "tsocial_labels.npy",

    "Undirected degree":
        "tsocial_undirected_degree.npy",

    "Different-label degree":
        "tsocial_different_label_degree.npy",

    "Local heterophily":
        "tsocial_local_heterophily.npy",

    "Reference train indices":
        "tsocial_reference_train_indices.npy",

    "Reference validation indices":
        "tsocial_reference_valid_indices.npy",

    "Reference test indices":
        "tsocial_reference_test_indices.npy"
}


print("\n===== REQUIRED FILE CHECK =====")


resolved_files = {}


for label, filename in required_core_files.items():

    path = os.path.join(
        ANALYSIS_DIR,
        filename
    )

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"Missing required output:\n"
            f"{label}\n"
            f"{path}"
        )

    resolved_files[
        filename
    ] = path

    print(
        "PASS:",
        filename
    )


# ============================================================
# 3. LOAD JSON RESULTS
# ============================================================

def load_json(filename):

    path = os.path.join(
        ANALYSIS_DIR,
        filename
    )

    with open(
        path,
        "r"
    ) as f:

        return json.load(f)


task3 = load_json(
    "tsocial_task3_structure.json"
)

task4 = load_json(
    "tsocial_task4_anomaly_origin.json"
)

task5 = load_json(
    "tsocial_task5_global_heterophily.json"
)

task6 = load_json(
    "tsocial_task6_local_heterophily.json"
)

task7 = load_json(
    "tsocial_task7_reference_split.json"
)

task8 = load_json(
    "tsocial_task8_sources_limitations.json"
)

task9 = load_json(
    "tsocial_task9_detailed_summary.json"
)

common_json = load_json(
    "tsocial_common_table_row.json"
)


print(
    "\nPASS: All JSON result files loaded."
)


# ============================================================
# 4. LOCKED EXPECTED VALUES
# ============================================================

EXPECTED = {

    "nodes":
        5_781_065,

    "normal":
        5_606_785,

    "anomaly":
        174_280,

    "features":
        10,

    "raw_edges":
        146_211_016,

    "undirected_edges":
        73_105_508,

    "homophilic_edges":
        45_610_885,

    "heterophilic_edges":
        27_494_623,

    "global_h":
        0.376095095324,

    "local_mean":
        0.122186690,

    "local_median":
        0.0,

    "local_std":
        0.291435701,

    "local_q1":
        0.0,

    "local_q3":
        0.0,

    "local_min":
        0.0,

    "local_max":
        1.0,

    "normal_local_mean":
        0.100309969,

    "anomaly_local_mean":
        0.825985563,

    "isolates":
        0,

    "train":
        2_312_426,

    "validation":
        1_144_650,

    "test":
        2_323_989,

    "source_sha256":
        (
            "8d577114cff12f7de35eda2974f17825"
            "ef9febe20c661e95e8d21072a6dfc8d0"
        )
}


# ============================================================
# 5. CORE NUMERICAL AUDIT
# ============================================================

print(
    "\n===== CORE NUMERICAL AUDIT ====="
)


assert (
    task5["nodes"]
    == EXPECTED["nodes"]
)

assert (
    task9["nodes"]
    == EXPECTED["nodes"]
)


assert (
    task9["normal_nodes"]
    == EXPECTED["normal"]
)

assert (
    task9["anomaly_nodes"]
    == EXPECTED["anomaly"]
)


assert (
    EXPECTED["normal"]
    + EXPECTED["anomaly"]
    == EXPECTED["nodes"]
)


assert (
    task9["features"]
    == EXPECTED["features"]
)


assert (
    task5[
        "raw_stored_dgl_edge_entries"
    ]
    == EXPECTED["raw_edges"]
)


assert (
    task5[
        "unique_undirected_nonself_edges"
    ]
    == EXPECTED["undirected_edges"]
)


assert (
    EXPECTED["raw_edges"]
    ==
    2 * EXPECTED["undirected_edges"]
)


print(
    "PASS: Nodes verified."
)

print(
    "PASS: Normal/anomaly counts verified."
)

print(
    "PASS: Feature dimension verified."
)

print(
    "PASS: Raw and comparable edge counts verified."
)


# ============================================================
# 6. STRUCTURAL AUDIT
# ============================================================

print(
    "\n===== STRUCTURAL AUDIT ====="
)


assert (
    task5["is_multigraph"]
    is False
)

assert (
    task5["self_loop_entries"]
    == 0
)

assert (
    task5["missing_reverse_entries"]
    == 0
)

assert (
    task5["fully_reciprocal"]
    is True
)

assert (
    task5["upper_direction_entries"]
    == EXPECTED["undirected_edges"]
)

assert (
    task5["lower_direction_entries"]
    == EXPECTED["undirected_edges"]
)


assert (
    task5[
        "homophilic_undirected_edges"
    ]
    == EXPECTED["homophilic_edges"]
)

assert (
    task5[
        "heterophilic_undirected_edges"
    ]
    == EXPECTED["heterophilic_edges"]
)


assert (
    EXPECTED["homophilic_edges"]
    + EXPECTED["heterophilic_edges"]
    == EXPECTED["undirected_edges"]
)


print(
    "PASS: No self-loops."
)

print(
    "PASS: No parallel directed edges."
)

print(
    "PASS: Complete reciprocity."
)

print(
    "PASS: Homophilic + heterophilic edges "
    "equal total comparable edges."
)


# ============================================================
# 7. GLOBAL HETEROPHILY AUDIT
# ============================================================

global_h = float(
    task5[
        "global_heterophily"
    ]
)


recalculated_global_h = (
    EXPECTED["heterophilic_edges"]
    /
    EXPECTED["undirected_edges"]
)


assert abs(
    global_h
    - recalculated_global_h
) < 1e-15


assert abs(
    global_h
    - EXPECTED["global_h"]
) < 1e-12


print(
    "\n===== GLOBAL HETEROPHILY AUDIT ====="
)

print(
    "Saved global H:",
    f"{global_h:.12f}"
)

print(
    "Recalculated global H:",
    f"{recalculated_global_h:.12f}"
)

print(
    "PASS: Global heterophily verified."
)


# ============================================================
# 8. LOCAL HETEROPHILY AUDIT
# ============================================================

print(
    "\n===== LOCAL HETEROPHILY AUDIT ====="
)


assert (
    task6["isolates"]
    == EXPECTED["isolates"]
)


assert abs(
    task6["mean"]
    - EXPECTED["local_mean"]
) < 1e-9


assert abs(
    task6["median"]
    - EXPECTED["local_median"]
) < 1e-12


assert abs(
    task6["population_std"]
    - EXPECTED["local_std"]
) < 1e-9


assert abs(
    task6["q1"]
    - EXPECTED["local_q1"]
) < 1e-12


assert abs(
    task6["q3"]
    - EXPECTED["local_q3"]
) < 1e-12


assert abs(
    task6["minimum"]
    - EXPECTED["local_min"]
) < 1e-12


assert abs(
    task6["maximum"]
    - EXPECTED["local_max"]
) < 1e-12


assert abs(
    task6["normal_mean_local_h"]
    - EXPECTED["normal_local_mean"]
) < 1e-9


assert abs(
    task6["anomaly_mean_local_h"]
    - EXPECTED["anomaly_local_mean"]
) < 1e-9


weighted_h = float(
    task6[
        "degree_weighted_local_heterophily"
    ]
)


assert abs(
    weighted_h
    - global_h
) < 1e-12


print(
    "PASS: Local-H summary verified."
)

print(
    "PASS: Population SD convention verified."
)

print(
    "PASS: No isolates."
)

print(
    "PASS: Weighted local H equals global H."
)


# ============================================================
# 9. VERIFY DEGREE ARRAYS DIRECTLY
# ============================================================

degree = np.load(
    resolved_files[
        "tsocial_undirected_degree.npy"
    ],
    mmap_mode="r"
)

different_degree = np.load(
    resolved_files[
        "tsocial_different_label_degree.npy"
    ],
    mmap_mode="r"
)

labels = np.load(
    resolved_files[
        "tsocial_labels.npy"
    ],
    mmap_mode="r"
)

local_h = np.load(
    resolved_files[
        "tsocial_local_heterophily.npy"
    ],
    mmap_mode="r"
)


assert (
    len(degree)
    == EXPECTED["nodes"]
)

assert (
    len(different_degree)
    == EXPECTED["nodes"]
)

assert (
    len(labels)
    == EXPECTED["nodes"]
)

assert (
    len(local_h)
    == EXPECTED["nodes"]
)


degree_sum = int(
    np.sum(
        degree,
        dtype=np.int64
    )
)

different_degree_sum = int(
    np.sum(
        different_degree,
        dtype=np.int64
    )
)


assert (
    degree_sum
    == 2 * EXPECTED["undirected_edges"]
)

assert (
    different_degree_sum
    == 2 * EXPECTED["heterophilic_edges"]
)


assert (
    np.count_nonzero(
        degree == 0
    )
    == 0
)


assert (
    np.count_nonzero(
        np.isnan(local_h)
    )
    == 0
)


print(
    "\nPASS: Degree arrays independently verified."
)

print(
    "PASS: Local-H array contains one defined "
    "value for every node."
)


# ============================================================
# 10. REFERENCE SPLIT AUDIT
# ============================================================

print(
    "\n===== REFERENCE SPLIT AUDIT ====="
)


assert (
    task7["train_nodes"]
    == EXPECTED["train"]
)

assert (
    task7["validation_nodes"]
    == EXPECTED["validation"]
)

assert (
    task7["test_nodes"]
    == EXPECTED["test"]
)


assert (
    EXPECTED["train"]
    + EXPECTED["validation"]
    + EXPECTED["test"]
    == EXPECTED["nodes"]
)


assert (
    task7["random_state"]
    == 2
)

assert (
    task7["temporal"]
    is False
)


train_idx = np.load(
    resolved_files[
        "tsocial_reference_train_indices.npy"
    ],
    mmap_mode="r"
)

valid_idx = np.load(
    resolved_files[
        "tsocial_reference_valid_indices.npy"
    ],
    mmap_mode="r"
)

test_idx = np.load(
    resolved_files[
        "tsocial_reference_test_indices.npy"
    ],
    mmap_mode="r"
)


assert (
    len(train_idx)
    == EXPECTED["train"]
)

assert (
    len(valid_idx)
    == EXPECTED["validation"]
)

assert (
    len(test_idx)
    == EXPECTED["test"]
)


print(
    "PASS: Train count verified."
)

print(
    "PASS: Validation count verified."
)

print(
    "PASS: Test count verified."
)

print(
    "PASS: Random state = 2."
)

print(
    "PASS: Split is non-temporal."
)


# ============================================================
# 11. PROVENANCE AUDIT
# ============================================================

print(
    "\n===== PROVENANCE AUDIT ====="
)


assert (
    task8["source_status"]
    == "Authors' official release used"
)

assert (
    task8["official_graph_sha256"]
    == EXPECTED["source_sha256"]
)

assert (
    task8["kaggle_mirror_used"]
    is False
)


assert (
    task4["dataset_origin"]
    == "Real-world"
)

assert (
    task4["anomaly_origin"]
    == "Real-world / non-injected"
)

assert (
    task4[
        "synthetic_anomalies_in_original_dataset"
    ]
    is False
)


print(
    "PASS: Official authors' release recorded."
)

print(
    "PASS: Official source SHA256 verified."
)

print(
    "PASS: Kaggle mirror excluded."
)

print(
    "PASS: Real-world/non-injected anomaly "
    "provenance verified."
)


# ============================================================
# 12. STRUCTURAL SEMANTICS AUDIT
# ============================================================

assert (
    task3["graph_type"]
    == "Homogeneous"
)

assert (
    task3["relation_structure"]
    == "Single-relational"
)

assert (
    task3["temporal_structure"]
    == "Static"
)


print(
    "\nPASS: Homogeneous classification verified."
)

print(
    "PASS: Single-relation classification verified."
)

print(
    "PASS: Static classification verified."
)


# ============================================================
# 13. COMMON-TABLE CSV / JSON AUDIT
# ============================================================

common_csv_path = resolved_files[
    "tsocial_common_table_row.csv"
]

common_df = pd.read_csv(
    common_csv_path
)


expected_columns = [

    "Dataset",
    "Domain",
    "Nodes",
    "Edges",
    "Features",
    "Fraud/anomaly nodes",
    "Normal nodes",
    "Fraud/anomaly %",
    "Imbalance ratio",
    "Graph type",
    "Node types",
    "Relation types",
    "Static/dynamic",
    "Organic/injected"
]


assert (
    list(common_df.columns)
    == expected_columns
)

assert (
    len(common_df)
    == 1
)


row = common_df.iloc[0]


assert (
    row["Dataset"]
    == "T-Social"
)

assert (
    int(row["Nodes"])
    == EXPECTED["nodes"]
)

assert (
    int(row["Features"])
    == EXPECTED["features"]
)

assert (
    int(row["Fraud/anomaly nodes"])
    == EXPECTED["anomaly"]
)

assert (
    int(row["Normal nodes"])
    == EXPECTED["normal"]
)


assert (
    row["Edges"]
    ==
    "73,105,508 unique undirected edges"
)


print(
    "\n===== COMMON TABLE AUDIT ====="
)

print(
    "PASS: Column structure verified."
)

print(
    "PASS: One T-Social row present."
)

print(
    "PASS: Common-table core values verified."
)


# ============================================================
# 14. CREATE FINAL HUMAN-READABLE SUMMARY
# ============================================================

FINAL_SUMMARY_PATH = os.path.join(
    ANALYSIS_DIR,
    "TSOCIAL_FINAL_SUMMARY.txt"
)


summary_text = f"""
T-SOCIAL FINAL DATASET AUDIT
============================

SOURCE
------
Dataset:
T-Social

Paper:
Tang et al. (2022)
Rethinking Graph Neural Networks for Anomaly Detection
ICML 2022

Source used:
Authors' official BWGNN release

Official extracted graph SHA256:
{EXPECTED["source_sha256"]}

Unofficial Kaggle mirror used:
No


CORE CHARACTERISTICS
--------------------
Nodes:
{EXPECTED["nodes"]:,}

Features:
{EXPECTED["features"]}

Normal nodes:
{EXPECTED["normal"]:,}

Anomaly nodes:
{EXPECTED["anomaly"]:,}

Anomaly percentage:
{EXPECTED["anomaly"] / EXPECTED["nodes"] * 100:.6f}%

Normal:Anomaly imbalance:
{EXPECTED["normal"] / EXPECTED["anomaly"]:.6f}:1


GRAPH STRUCTURE
---------------
Graph type:
Homogeneous

Relation structure:
Single-relational

Semantic node type:
Anonymized social-network account

Semantic relation:
Social friendship

Temporal structure:
Static

Raw stored DGL edge entries:
{EXPECTED["raw_edges"]:,}

Self-loops:
0

Parallel directed edges:
No

Missing reverse edges:
0

Unique undirected non-self edges:
{EXPECTED["undirected_edges"]:,}


GLOBAL HETEROPHILY
------------------
Homophilic edges:
{EXPECTED["homophilic_edges"]:,}

Heterophilic edges:
{EXPECTED["heterophilic_edges"]:,}

Global heterophily:
{global_h:.12f}

Global heterophily percentage:
{global_h * 100:.6f}%


LOCAL HETEROPHILY
-----------------
Mean:
{task6["mean"]:.9f}

Median:
{task6["median"]:.9f}

Population SD:
{task6["population_std"]:.9f}

Q1:
{task6["q1"]:.9f}

Q3:
{task6["q3"]:.9f}

Minimum:
{task6["minimum"]:.9f}

Maximum:
{task6["maximum"]:.9f}

Normal-node mean:
{task6["normal_mean_local_h"]:.9f}

Anomaly-node mean:
{task6["anomaly_mean_local_h"]:.9f}

Isolates:
{task6["isolates"]:,}

Degree-weighted local H:
{weighted_h:.12f}


REFERENCE SPLIT
---------------
Type:
Random stratified node split

Random state:
2

Train:
{task7["train_nodes"]:,}
({task7["train_percent"]:.6f}%)

Validation:
{task7["validation_nodes"]:,}
({task7["validation_percent"]:.6f}%)

Test:
{task7["test_nodes"]:,}
({task7["test_percent"]:.6f}%)

Train normal/anomaly:
{task7["train_normal"]:,} / {task7["train_anomaly"]:,}

Validation normal/anomaly:
{task7["validation_normal"]:,} / {task7["validation_anomaly"]:,}

Test normal/anomaly:
{task7["test_normal"]:,} / {task7["test_anomaly"]:,}


ANOMALY PROVENANCE
------------------
Dataset origin:
Real-world

Label source:
Human-expert annotation

Positive class:
Anomalous account

Examples:
Fraud, money laundering, online gambling

Synthetic/injected anomalies in original dataset:
No


EDGE COUNT CONVENTION
---------------------
Use 73,105,508 as the common-table edge count.

The original DGL graph stores 146,211,016 directed entries.
The graph was verified to contain:

- zero self-loops,
- no parallel directed edges,
- complete reciprocity.

Therefore the raw DGL representation contains two directed
entries for each of 73,105,508 unique undirected friendships.


FINAL STATUS
------------
T-Social dataset analysis PASSED all final consistency checks.
"""


with open(
    FINAL_SUMMARY_PATH,
    "w"
) as f:

    f.write(
        summary_text.strip()
        + "\n"
    )


print(
    "\nPASS: Human-readable final summary created."
)


# ============================================================
# 15. CREATE FINAL AUDIT JSON
# ============================================================

FINAL_AUDIT_PATH = os.path.join(
    ANALYSIS_DIR,
    "tsocial_final_audit.json"
)


final_audit = {

    "dataset":
        "T-Social",

    "status":
        "PASS",

    "authoritative_source":
        "Official BWGNN authors' release",

    "official_source_sha256":
        EXPECTED["source_sha256"],

    "nodes":
        EXPECTED["nodes"],

    "features":
        EXPECTED["features"],

    "normal_nodes":
        EXPECTED["normal"],

    "anomaly_nodes":
        EXPECTED["anomaly"],

    "raw_stored_dgl_entries":
        EXPECTED["raw_edges"],

    "comparable_unique_undirected_edges":
        EXPECTED["undirected_edges"],

    "global_heterophily":
        global_h,

    "local_h_mean":
        task6["mean"],

    "normal_mean_local_h":
        task6[
            "normal_mean_local_h"
        ],

    "anomaly_mean_local_h":
        task6[
            "anomaly_mean_local_h"
        ],

    "isolates":
        task6["isolates"],

    "reference_split": {

        "type":
            task7["split_type"],

        "random_state":
            task7["random_state"],

        "train":
            task7["train_nodes"],

        "validation":
            task7["validation_nodes"],

        "test":
            task7["test_nodes"]
    },

    "final_validation": {

        "class_counts":
            "PASS",

        "edge_structure":
            "PASS",

        "global_heterophily":
            "PASS",

        "local_heterophily":
            "PASS",

        "weighted_global_local_identity":
            "PASS",

        "reference_split":
            "PASS",

        "source_provenance":
            "PASS",

        "common_table_row":
            "PASS"
    }
}


with open(
    FINAL_AUDIT_PATH,
    "w"
) as f:

    json.dump(
        final_audit,
        f,
        indent=4
    )


print(
    "PASS: Final audit JSON created."
)


# ============================================================
# 16. CREATE MANIFEST
# ============================================================

def sha256_for_manifest(
    path,
    chunk_size=16 * 1024 * 1024
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# Files to package.
# The original 4.11 GB graph and legacy environment are
# intentionally excluded.

package_filenames = [

    "tsocial_task3_structure.json",
    "tsocial_task4_anomaly_origin.json",
    "tsocial_task5_global_heterophily.json",
    "tsocial_task6_local_heterophily.json",
    "tsocial_task7_reference_split.json",
    "tsocial_task8_sources_limitations.json",
    "tsocial_common_table_row.csv",
    "tsocial_common_table_row.json",
    "tsocial_task9_detailed_summary.json",

    "tsocial_labels.npy",
    "tsocial_undirected_degree.npy",
    "tsocial_different_label_degree.npy",
    "tsocial_local_heterophily.npy",

    "tsocial_reference_train_indices.npy",
    "tsocial_reference_valid_indices.npy",
    "tsocial_reference_test_indices.npy",

    "TSOCIAL_FINAL_SUMMARY.txt",
    "tsocial_final_audit.json"
]


MANIFEST_PATH = os.path.join(
    ANALYSIS_DIR,
    "tsocial_manifest.csv"
)


manifest_rows = []


print(
    "\n===== BUILDING SHA256 MANIFEST ====="
)


for filename in package_filenames:

    path = os.path.join(
        ANALYSIS_DIR,
        filename
    )

    if not os.path.isfile(path):

        raise FileNotFoundError(
            f"Package file missing:\n{path}"
        )


    size = os.path.getsize(
        path
    )


    checksum = sha256_for_manifest(
        path
    )


    manifest_rows.append({

        "filename":
            filename,

        "size_bytes":
            size,

        "sha256":
            checksum
    })


    print(
        "PASS:",
        filename,
        f"({size:,} bytes)"
    )


manifest_df = pd.DataFrame(
    manifest_rows
)


manifest_df.to_csv(
    MANIFEST_PATH,
    index=False
)


print(
    "\nPASS: Manifest created:"
)

print(
    MANIFEST_PATH
)


# Add manifest itself to ZIP
package_filenames.append(
    "tsocial_manifest.csv"
)


# ============================================================
# 17. CREATE FINAL ZIP PACKAGE
# ============================================================

ZIP_PATH = (
    "/kaggle/working/"
    "T_Social_Final_Analysis_Package.zip"
)


print(
    "\n===== CREATING FINAL ZIP ====="
)

print(
    "ZIP:"
)

print(
    ZIP_PATH
)


with zipfile.ZipFile(
    ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as z:

    for filename in package_filenames:

        source_path = os.path.join(
            ANALYSIS_DIR,
            filename
        )


        z.write(
            source_path,
            arcname=filename
        )


        print(
            "Added:",
            filename
        )


# ============================================================
# 18. VERIFY ZIP CONTENTS
# ============================================================

if not os.path.isfile(
    ZIP_PATH
):

    raise FileNotFoundError(
        "Final ZIP package was not created."
    )


with zipfile.ZipFile(
    ZIP_PATH,
    "r"
) as z:

    bad_member = z.testzip()

    if bad_member is not None:

        raise RuntimeError(
            "ZIP integrity failure in:\n"
            f"{bad_member}"
        )


    zip_members = z.namelist()


expected_zip_members = set(
    package_filenames
)


assert (
    set(zip_members)
    == expected_zip_members
)


print(
    "\nPASS: ZIP integrity check passed."
)

print(
    "PASS: ZIP contains every expected artifact."
)


# ============================================================
# 19. FINAL PACKAGE SIZE
# ============================================================

zip_size = os.path.getsize(
    ZIP_PATH
)


print(
    "\nFinal ZIP size:",
    f"{zip_size:,}",
    "bytes"
)


# ============================================================
# 20. FINAL AUDIT REPORT
# ============================================================

print(
    "\n======================================"
)

print(
    "T-SOCIAL TASK 10 COMPLETE"
)

print(
    "======================================"
)


print(
    "\nFINAL STATUS: PASS"
)


print(
    "\nAuthoritative source:"
)

print(
    "Official BWGNN authors' T-Social release"
)


print(
    "\nNodes:",
    f"{EXPECTED['nodes']:,}"
)

print(
    "Comparable edges:",
    f"{EXPECTED['undirected_edges']:,}"
)

print(
    "Features:",
    EXPECTED["features"]
)

print(
    "Normal nodes:",
    f"{EXPECTED['normal']:,}"
)

print(
    "Anomaly nodes:",
    f"{EXPECTED['anomaly']:,}"
)

print(
    "Global heterophily:",
    f"{global_h:.9f}"
)

print(
    "Local H mean:",
    f"{task6['mean']:.9f}"
)

print(
    "Normal-node mean local H:",
    f"{task6['normal_mean_local_h']:.9f}"
)

print(
    "Anomaly-node mean local H:",
    f"{task6['anomaly_mean_local_h']:.9f}"
)

print(
    "Isolates:",
    task6["isolates"]
)


print(
    "\nReference split:"
)

print(
    f"Train      = {task7['train_nodes']:,}"
)

print(
    f"Validation = {task7['validation_nodes']:,}"
)

print(
    f"Test       = {task7['test_nodes']:,}"
)


print(
    "\nFinal common-table CSV:"
)

print(
    os.path.join(
        ANALYSIS_DIR,
        "tsocial_common_table_row.csv"
    )
)


print(
    "\nFinal human-readable summary:"
)

print(
    FINAL_SUMMARY_PATH
)


print(
    "\nFinal reproducibility ZIP:"
)

print(
    ZIP_PATH
)


print(
    "\nT-Social analysis is complete."
)

===== T-SOCIAL TASK 10 =====

Analysis directory:
/kaggle/working/tsocial_analysis

===== REQUIRED FILE CHECK =====
PASS: tsocial_task3_structure.json
PASS: tsocial_task4_anomaly_origin.json
PASS: tsocial_task5_global_heterophily.json
PASS: tsocial_task6_local_heterophily.json
PASS: tsocial_task7_reference_split.json
PASS: tsocial_task8_sources_limitations.json
PASS: tsocial_common_table_row.csv
PASS: tsocial_common_table_row.json
PASS: tsocial_task9_detailed_summary.json
PASS: tsocial_labels.npy
PASS: tsocial_undirected_degree.npy
PASS: tsocial_different_label_degree.npy
PASS: tsocial_local_heterophily.npy
PASS: tsocial_reference_train_indices.npy
PASS: tsocial_reference_valid_indices.npy
PASS: tsocial_reference_test_indices.npy

PASS: All JSON result files loaded.

===== CORE NUMERICAL AUDIT =====
PASS: Nodes verified.
PASS: Normal/anomaly counts verified.
PASS: Feature dimension verified.
PASS: Raw and comparable edge counts verified.

===== STRUCTURAL AUDIT =====
PASS: No self-loop